In [ ]:
import pandas as pd
import numpy as np
import json
import os
import math
import calendar
from datetime import datetime, date

# =============================================================
#  Smart APS V9  —  Generalised Multi-Section Scheduler
#
#  To use for a different section (e.g. HT, ST) change only:
#    SECTION 1  — planning date
#    SECTION 3  — file paths and sheet names
#    SECTION 2  — parameters (if different for the section)
#
#  All logic, column names, and constraints remain identical.
#
#  Constraints implemented:
#  ─────────────────────────────────────────────────────────────
#  1. Fixed machine constraint
#     - Phase A (inv < SAFETY_DAYS): machine dedicated to fixed
#       part only, full hours (23h if daily_indent/rate > 20h)
#     - Phase B (inv ≥ SAFETY_DAYS): fixed part runs for
#       max(MIN_RUN_HOURS, daily_indent/rate) first, then
#       remaining hours available to any compatible part
#     - Two fixed parts on same machine: pick one per day based
#       on lowest days-coverage first, tie → highest hrs_needed
#     - Cross-machine spill: only if fixed machine assigned first
#       AND indent still short AND tools > 1
#
#  2. Runner priority enforcement
#     - Runner with inv < 2 × daily_indent: displacement eligible
#       Can displace Strangers AND Repeaters with ≥ 1 day stock
#
#  3. Repeater zero-inventory protection
#     - Repeater with inv = 0: displacement eligible
#       Can displace Strangers AND Repeaters with ≥ 1 day stock
#
#  4. Multi-part machine planning
#     - Cover daily indent of each part first (highest priority
#       score first), then extend both toward OPD cap with
#       remaining hours
#
#  5. Terminal constraint (inventory-based)
#  6. Colour purge penalty (10 min for colour change)
#  7. Tool-aware assignment (minimum tools principle)
#  8. CO stagger (quantity-based + serial queue)
#  9. 22h utilisation enforcer with 90% floor
# =============================================================

# =============================================================
# SECTION 1 — DAILY SETTINGS
# =============================================================

PLANNING_DATE = date(2026, 4, 9)
INDENT_MONTH  = date(2026, 4, 1)

# =============================================================
# SECTION 2 — PARAMETERS
# =============================================================

AVAILABLE_HOURS      = 22       # normal shift hours
EXTENDED_HOURS       = 23       # used only for fixed parts when needed > 20h
EXTENDED_THRESHOLD   = 20       # hours: if daily_indent/rate > this → 23h
MIN_RUN_HOURS        = 4
MACHINE_STATE_FILE   = "machine_state.json"

MIN_DAILY_INDENT     = 150
MIN_INDENT_HOURS     = 4.0

SAFETY_DAYS  = 3    # buffer target — fixed machine released above this
TARGET_DAYS  = 5    # inventory ceiling

OPD_SCENARIO_0 = 1.5
OPD_SCENARIO_1 = 3.0
OPD_SCENARIO_2 = 4.0
OPD_SCENARIO_3 = 5.0

W_URGENCY  = 0.55
W_CATEGORY = 0.25
W_INDENT   = 0.20

UTIL_TARGET_PCT  = 90.0    # minimum utilisation floor
COLOR_PURGE_HRS  = 10 / 60.0    # 10 min purge for colour change

# Displacement thresholds
RUNNER_DISPLACE_DAYS   = 2.0    # Runner with inv < N × daily eligible for displacement
REPEATER_VICTIM_DAYS   = 1.0    # Repeater with inv >= N days can be displaced

# =============================================================
# SECTION 3 — FILE PATHS AND SHEET NAMES
# ─────────────────────────────────────────────────────────────
# To adapt for another section change the paths and sheet names
# below. Everything else is automatic.
# =============================================================

# Input files
book_path       = "C:/Users/Ex0164/Book1.xlsx"
matrix_path     = "C:/Users/Ex0164/Important codes/compatibility_matrix.xlsx"
changeover_path = "D:/Tushar/TOOL FIX/Unique_Machines_vt.xlsx"
terminal_path   = "C:/Users/Ex0164/Important codes/terminals and raw marterial - vt.xlsx"

# Sheet names inside the input files
SECTION_LABEL          = "VT"                    # label used in output sheet names
PARTS_SHEET            = "VT"                    # sheet in book_path
MATRIX_SHEET           = "VT_Matrix"             # sheet in matrix_path
CHANGEOVER_SHEET       = "VT_Changeover"         # sheet in changeover_path
MACHINE_COUNT_SHEET    = "VT_Machine_Part_Count" # sheet in matrix_path
FIXED_SHEET            = "VT_Fixed"              # sheet in matrix_path
TERMINALS_SHEET        = "VT_Terminals"          # sheet in terminal_path
TERMINAL_INV_SHEET     = "VT_Terminal_Inventory" # sheet in terminal_path

# Column names in PARTS_SHEET
COL_PART       = "Part"
COL_CYCLETIME  = "Cycle time"
COL_CAVITY     = "Cavity"
COL_INVENTORY  = "Inventory"
COL_INDENT     = "Indent"
COL_TOOLS      = "Tools"
COL_COLOR      = "Color"
COL_CATEGORY   = "Category"    # Runner / Repeater / Stranger

# Column name in CHANGEOVER_SHEET
COL_CO_MACHINE = "Unique Machines"
COL_CO_MINUTES = "Changeover time"

# Output file
output_path = (
    f"Smart_APS_V9_{SECTION_LABEL}_Plan_"
    f"{PLANNING_DATE.strftime('%Y%m%d')}.xlsx"
)

# =============================================================
# SECTION 4 — WORKING DAYS
# =============================================================

def compute_working_days(ref_date):
    year  = ref_date.year
    month = ref_date.month
    total = calendar.monthrange(year, month)[1]
    sundays = sum(
        1 for d in range(1, total + 1)
        if date(year, month, d).weekday() == 6
    )
    return total - sundays, total, sundays

WORKING_DAYS, TOTAL_DAYS, SUNDAY_COUNT = compute_working_days(INDENT_MONTH)

print(f"\n{'='*65}")
print(f"  Smart APS V9  —  Section: {SECTION_LABEL}")
print(f"  Planning date : {PLANNING_DATE}")
print(f"  Indent month  : {INDENT_MONTH.strftime('%B %Y')}")
print(f"  Working days  : {WORKING_DAYS}  ({TOTAL_DAYS} − {SUNDAY_COUNT} Sundays)")
print(f"  Safety floor  : {SAFETY_DAYS} days  |  Target : {TARGET_DAYS} days")
print(f"{'='*65}\n")

# =============================================================
# SECTION 5 — LOAD DATA
# =============================================================

print("Loading data...")
parts_raw            = pd.read_excel(book_path,       sheet_name=PARTS_SHEET)
matrix_df            = pd.read_excel(matrix_path,     sheet_name=MATRIX_SHEET)
co_raw               = pd.read_excel(changeover_path, sheet_name=CHANGEOVER_SHEET)
machine_count_raw    = pd.read_excel(matrix_path,     sheet_name=MACHINE_COUNT_SHEET)

try:
    fixed_raw = pd.read_excel(matrix_path, sheet_name=FIXED_SHEET)
    print(f"  Fixed sheet loaded      : {len(fixed_raw)} rows")
except Exception as _fe:
    fixed_raw = None
    print(f"  WARNING: '{FIXED_SHEET}' not found — fixed constraint disabled")

try:
    terminals_raw     = pd.read_excel(terminal_path, sheet_name=TERMINALS_SHEET)
    terminal_inv_raw  = pd.read_excel(terminal_path, sheet_name=TERMINAL_INV_SHEET)
    print(f"  Terminal data loaded from : {terminal_path}")
except FileNotFoundError:
    terminals_raw = terminal_inv_raw = None
    print(f"  WARNING: terminal file not found — terminal constraint disabled")
except Exception as _te:
    terminals_raw = terminal_inv_raw = None
    print(f"  WARNING: terminal load error ({_te}) — constraint disabled")

# =============================================================
# SECTION 6 — PARSE PARTS SHEET
# =============================================================

def find_col(df, name, sheet):
    m = next((c for c in df.columns if str(c).strip().lower() == name.lower()), None)
    if m is None:
        raise ValueError(
            f"Column '{name}' not found in '{sheet}'.\n"
            f"Available: {list(df.columns)}"
        )
    return m

c_part  = find_col(parts_raw, COL_PART,      PARTS_SHEET)
c_ct    = find_col(parts_raw, COL_CYCLETIME, PARTS_SHEET)
c_cav   = find_col(parts_raw, COL_CAVITY,    PARTS_SHEET)
c_inv   = find_col(parts_raw, COL_INVENTORY, PARTS_SHEET)
c_ind   = find_col(parts_raw, COL_INDENT,    PARTS_SHEET)
c_tools = find_col(parts_raw, COL_TOOLS,     PARTS_SHEET)
c_color = find_col(parts_raw, COL_COLOR,     PARTS_SHEET)

data = parts_raw[
    parts_raw[c_part].notna() &
    (parts_raw[c_part].astype(str).str.strip() != "")
].copy()
data = data.drop_duplicates(subset=c_part).copy()
data["Material"] = data[c_part].astype(str).str.strip()
data["_ct"]      = pd.to_numeric(data[c_ct], errors="coerce").replace(0, np.nan)
data["Rate"]     = 3600 / data["_ct"]

data_valid     = data[data["Rate"].notna()].copy()
data_zero_rate = data[data["Rate"].isna()].copy()
print(f"  Parts in sheet          : {len(data)}")
print(f"  Parts with valid rate   : {len(data_valid)}")

# =============================================================
# SECTION 7 — LOOKUP DICTIONARIES
# =============================================================

def safe_dict(df, key_col, val_col, default=0.0):
    return {
        str(k).strip(): (default if pd.isna(v) else float(v))
        for k, v in zip(df[key_col], df[val_col])
        if pd.notna(k) and str(k).strip() != ""
    }

inventory      = safe_dict(data,       "Material", c_inv)
rate           = safe_dict(data_valid, "Material", "Rate")
indent_monthly = safe_dict(data,       "Material", c_ind)

tools_available = {}
part_color      = {}
part_category   = {}

for _, row in data.iterrows():
    p = str(row["Material"]).strip()

    v = row[c_tools]
    tools_available[p] = (
        max(1, int(float(v))) if pd.notna(v) and str(v).strip() != "" else 1
    )
    c = row[c_color]
    part_color[p] = (
        str(c).strip().upper()
        if pd.notna(c) and str(c).strip() not in ("", "nan") else "UNKNOWN"
    )
    # Category — try column, default Stranger
    cat_col = next(
        (col for col in data.columns
         if str(col).strip().lower() == COL_CATEGORY.lower()), None
    )
    if cat_col:
        cat_val = str(row[cat_col]).strip().capitalize() if pd.notna(row[cat_col]) else "Stranger"
        part_category[p] = (
            cat_val if cat_val in ("Runner", "Repeater", "Stranger") else "Stranger"
        )
    else:
        part_category[p] = "Stranger"

color_groups = {}
for p, c in part_color.items():
    color_groups.setdefault(c, []).append(p)
print(f"  Distinct colours        : {len(color_groups)}")

indent_daily = {
    p: round(qty / WORKING_DAYS, 4) for p, qty in indent_monthly.items()
}
today_target_qty = {
    p: max(0.0, indent_daily.get(p, 0.0) - inventory.get(p, 0.0))
    for p in indent_monthly
}

CATEGORY_SCORE = {"Runner": 100, "Repeater": 60, "Stranger": 20}

# =============================================================
# SECTION 7A — FIXED MACHINE CONSTRAINT
# =============================================================

def build_fixed_machine_dicts(df):
    """
    Reads Fixed sheet: Machine | Part_col_1 | Part_col_2 | ...
    Returns:
      part_fixed_machine  : {part → machine}
      machine_fixed_parts : {machine → [part, ...]}
    """
    pfm, mfp = {}, {}
    if df is None or df.empty:
        return pfm, mfp

    machine_col = next(
        (c for c in df.columns if str(c).strip().lower() == "machine"), None
    )
    if machine_col is None:
        print("  WARNING: Fixed sheet has no 'Machine' column — disabled")
        return pfm, mfp

    part_cols = [c for c in df.columns if str(c).strip().lower() != "machine"]
    if not part_cols:
        print("  WARNING: Fixed sheet has no part columns — disabled")
        return pfm, mfp

    for _, row in df.iterrows():
        machine = row[machine_col]
        if pd.isna(machine) or str(machine).strip() == "":
            continue
        m = str(machine).strip()
        for col in part_cols:
            val = row[col]
            if pd.isna(val) or str(val).strip() in ("", "nan"):
                continue
            p = str(val).strip()
            if p in pfm:
                print(f"  WARNING: Part '{p}' in Fixed sheet twice — keeping {pfm[p]}")
                continue
            pfm[p] = m
            mfp.setdefault(m, []).append(p)
    return pfm, mfp


part_fixed_machine, machine_fixed_parts = build_fixed_machine_dicts(fixed_raw)
print(f"  Fixed machine mappings  : {len(part_fixed_machine)} parts")
for m, pts in sorted(machine_fixed_parts.items()):
    print(f"    {m:<25} ← {', '.join(pts)}")

# =============================================================
# SECTION 7B — FIXED MACHINE PHASE DETERMINATION
# =============================================================

def fixed_machine_phase(machine, current_inv):
    """
    Returns phase for a fixed machine based on start-of-day inventory.

    Phase A: ALL fixed parts on this machine have inv < SAFETY_DAYS
             → machine dedicated, run only fixed parts
    Phase B: ALL fixed parts have inv >= SAFETY_DAYS
             → run fixed part for min hours, rest open to scheduling
    Mixed  : some below, some above → treat as Phase A (conservative)

    Returns: "A" or "B"
    Also returns: chosen_part (the part to run today in Phase A)
    """
    fixed_parts = machine_fixed_parts.get(machine, [])
    if not fixed_parts:
        return "B", None

    all_healthy = all(
        current_inv.get(p, 0) >= SAFETY_DAYS * indent_daily.get(p, 0)
        for p in fixed_parts
    )
    if all_healthy:
        return "B", None

    # Phase A — pick which part to run today
    # Primary sort: lowest days-coverage (most urgent)
    # Tiebreaker: highest hours-needed for daily indent
    def sort_key(p):
        daily   = indent_daily.get(p, 0)
        inv     = current_inv.get(p, 0)
        r       = rate.get(p, 1)
        days_cov = inv / daily if daily > 0 else 999
        hrs_needed = daily / r if r > 0 else 0
        return (round(days_cov, 4), -round(hrs_needed, 4))

    chosen = sorted(fixed_parts, key=sort_key)[0]
    return "A", chosen


def fixed_part_hours(part):
    """
    Hours needed to produce exactly daily indent for a fixed part.
    If this exceeds EXTENDED_THRESHOLD, the part gets EXTENDED_HOURS
    (23h) instead of AVAILABLE_HOURS (22h) — but only for its own run.
    Returns (run_hours, machine_cap_for_others)
      run_hours          = actual hours for fixed part
      machine_cap_others = AVAILABLE_HOURS (22h) minus run_hours,
                           capped at AVAILABLE_HOURS for other parts
    """
    daily = indent_daily.get(part, 0)
    r     = rate.get(part, 1)
    if daily <= 0 or r <= 0:
        return MIN_RUN_HOURS, AVAILABLE_HOURS - MIN_RUN_HOURS

    hrs_needed = daily / r
    run_hrs    = max(MIN_RUN_HOURS, hrs_needed)

    if hrs_needed > EXTENDED_THRESHOLD:
        # Part gets up to EXTENDED_HOURS for its own run
        run_hrs = min(run_hrs, EXTENDED_HOURS)
    else:
        run_hrs = min(run_hrs, AVAILABLE_HOURS)

    # Remaining hours for other parts (capped at AVAILABLE_HOURS)
    remaining_for_others = max(0.0, AVAILABLE_HOURS - run_hrs)
    return round(run_hrs, 4), round(remaining_for_others, 4)

# =============================================================
# SECTION 7C — TERMINAL CONSTRAINT
# =============================================================

def _build_part_terminals(df):
    result = {}
    if df is None or df.empty:
        return result
    part_col = next(
        (c for c in df.columns
         if str(c).strip().lower() in ("part", "material")), None
    )
    if part_col is None:
        return result
    term_cols = [
        c for c in df.columns
        if str(c).strip().lower() not in ("part", "material")
    ]
    for _, row in df.iterrows():
        part = row[part_col]
        if pd.isna(part) or str(part).strip() == "":
            continue
        p     = str(part).strip()
        terms = [
            str(row[col]).strip().upper()
            for col in term_cols
            if pd.notna(row[col]) and str(row[col]).strip() not in ("", "nan")
        ]
        if terms:
            result[p] = terms
    return result


def _build_terminal_inv(df):
    result = {}
    if df is None or df.empty:
        return result
    t_col = next((c for c in df.columns if str(c).strip().lower() == "terminal"), None)
    i_col = next((c for c in df.columns if str(c).strip().lower() == "inventory"), None)
    if t_col is None or i_col is None:
        return result
    for _, row in df.iterrows():
        t = row[t_col]
        if pd.isna(t) or str(t).strip() == "":
            continue
        key = str(t).strip().upper()
        try:
            result[key] = float(row[i_col]) if pd.notna(row[i_col]) else 0.0
        except (ValueError, TypeError):
            result[key] = 0.0
    return result


part_terminals  = _build_part_terminals(terminals_raw)
terminal_status = _build_terminal_inv(terminal_inv_raw)

if terminal_status:
    zero_t = sum(1 for v in terminal_status.values() if v <= 0)
    print(f"  Terminals loaded        : {len(terminal_status)}  |  {zero_t} at zero")
else:
    print(f"  Terminals               : no data — constraint inactive")


def terminal_blocked(part):
    required = part_terminals.get(part, [])
    if not required:
        return False, ""
    zero = [t for t in required if terminal_status.get(t, 0) <= 0]
    if zero:
        details = ", ".join(
            f"{t}(inv={terminal_status.get(t,0):.0f})" for t in sorted(zero)
        )
        return True, (
            f"Terminal(s) zero stock: {details}  "
            f"(requires: {', '.join(required)})"
        )
    return False, ""

# =============================================================
# SECTION 7D — SKIP RULES
# =============================================================

def should_skip(part):
    daily   = indent_daily.get(part, 0.0)
    monthly = indent_monthly.get(part, 0.0)
    r       = rate.get(part, 1.0)

    if daily <= MIN_DAILY_INDENT:
        return True, f"Daily indent {daily:.2f} ≤ {MIN_DAILY_INDENT}"
    indent_hrs = monthly / r if r > 0 else 0.0
    if indent_hrs <= MIN_INDENT_HOURS:
        return True, f"Monthly hrs {indent_hrs:.2f}h ≤ {MIN_INDENT_HOURS}h"
    inv = inventory.get(part, 0.0)
    if daily > 0 and inv >= TARGET_DAYS * daily:
        return True, (
            f"Inv ({inv:.0f}) ≥ {TARGET_DAYS}-day target "
            f"({TARGET_DAYS * daily:.0f})"
        )
    t_blk, t_rsn = terminal_blocked(part)
    if t_blk:
        return True, t_rsn
    return False, ""

# =============================================================
# SECTION 7E — CHANGEOVER TIMES
# =============================================================

def build_changeover_dict(co_df):
    co = {}
    for _, row in co_df.iterrows():
        m = str(row[COL_CO_MACHINE]).strip()
        v = row[COL_CO_MINUTES]
        if pd.notna(v) and m:
            co[m] = float(v) / 60.0
    return co

vt_changeover          = build_changeover_dict(co_raw)
DEFAULT_CHANGEOVER_HRS = 40 / 60.0

# =============================================================
# SECTION 7F — MACHINE PART COUNT
# =============================================================

def build_machine_part_count(df):
    mpc = {}
    m_col = next((c for c in df.columns if str(c).strip().lower() == "machine"),   None)
    c_col = next((c for c in df.columns if str(c).strip().lower() == "part_count"), None)
    if m_col is None or c_col is None:
        return {}
    for _, row in df.iterrows():
        m = str(row[m_col]).strip()
        v = row[c_col]
        if m and pd.notna(v):
            try:
                mpc[m] = int(float(v))
            except (ValueError, TypeError):
                pass
    return mpc

machine_part_count = build_machine_part_count(machine_count_raw)
max_part_count     = max(machine_part_count.values(), default=1) or 1
print(f"  Machine part counts     : {len(machine_part_count)} machines")

# =============================================================
# SECTION 8 — MACHINE STATE
# =============================================================

def load_machine_state():
    if os.path.exists(MACHINE_STATE_FILE):
        try:
            content = open(MACHINE_STATE_FILE).read().strip()
            if not content:
                os.remove(MACHINE_STATE_FILE)
                return {}
            state = json.loads(content)
            if not isinstance(state, dict):
                os.remove(MACHINE_STATE_FILE)
                return {}
            print(f"  Machine state loaded    : {len(state)} machines")
            return state
        except Exception as e:
            print(f"  Machine state error ({e}) — first run")
            os.remove(MACHINE_STATE_FILE)
            return {}
    print(f"  Machine state           : FIRST RUN")
    return {}

def save_machine_state(state):
    with open(MACHINE_STATE_FILE, "w") as f:
        json.dump({m: p for m, p in state.items() if p is not None}, f, indent=2)
    print(f"\n  Machine state saved → '{MACHINE_STATE_FILE}'")

machine_state = load_machine_state()

# =============================================================
# SECTION 9 — COMPATIBILITY MATRIX
# =============================================================

def build_compatibility(matrix):
    machines = matrix.columns[1:].tolist()
    compat   = {}
    for _, row in matrix.iterrows():
        part = row["Part"]
        for m in machines:
            if row[m] == 1:
                compat.setdefault(part, []).append(m)
    return compat, machines

compat, all_machines = build_compatibility(matrix_df)

# =============================================================
# SECTION 10 — SCENARIO CLASSIFIER
# =============================================================

def classify_scenario(parts):
    cov = []
    for p in parts:
        daily = indent_daily.get(p, 0)
        skip, _ = should_skip(p)
        if skip or daily == 0:
            continue
        cov.append(inventory.get(p, 0) / daily)
    if not cov:
        return 3, "SCENARIO 3 — No active parts"
    n        = len(cov)
    critical = sum(1 for c in cov if c < 1)
    low      = sum(1 for c in cov if c < SAFETY_DAYS)
    if critical == n:
        return 0, f"SCENARIO 0 — ALL {n} parts critical"
    elif critical > 0:
        return 1, f"SCENARIO 1 — {critical}/{n} critical"
    elif low > 0:
        return 2, f"SCENARIO 2 — {low}/{n} below {SAFETY_DAYS}-day floor"
    else:
        return 3, f"SCENARIO 3 — All {n} healthy"

def opd_cap(scenario_id):
    return min(
        {0: OPD_SCENARIO_0, 1: OPD_SCENARIO_1,
         2: OPD_SCENARIO_2, 3: OPD_SCENARIO_3}.get(scenario_id, OPD_SCENARIO_2),
        TARGET_DAYS,
    )

# =============================================================
# SECTION 11 — PRIORITY SCORING
# =============================================================

def compute_priority_scores(active_parts):
    rows = []
    for p in active_parts:
        inv      = inventory.get(p, 0)
        daily    = indent_daily.get(p, 0)
        cat      = part_category.get(p, "Stranger")
        days_cov = inv / daily if daily > 0 else 999.0
        gap      = min(1.0, max(0.0, SAFETY_DAYS - days_cov) / SAFETY_DAYS)
        rows.append({"part": p, "inv": inv, "daily": daily,
                     "days_cov": days_cov, "cat": cat, "gap": gap})
    if not rows:
        return {}, []
    max_daily = max(r["daily"] for r in rows) or 1.0
    scores, score_rows = {}, []
    for r in rows:
        p     = r["part"]
        urg   = r["gap"] * 100
        cat_s = CATEGORY_SCORE.get(r["cat"], 20)
        ind_s = (r["daily"] / max_daily) * 100
        fs    = W_URGENCY * urg + W_CATEGORY * cat_s + W_INDENT * ind_s
        scores[p] = round(fs, 2)
        score_rows.append({
            "Part":           p,
            "Category":       r["cat"],
            "Color":          part_color.get(p, "UNKNOWN"),
            "Fixed_Machine":  part_fixed_machine.get(p, "—"),
            "Tools":          tools_available.get(p, 1),
            "Inventory_Now":  round(r["inv"], 0),
            "Daily_Indent":   round(r["daily"], 2),
            "Days_Coverage":  round(r["days_cov"], 2),
            "Buffer_Status":  (
                "CRITICAL"     if r["days_cov"] < 1 else
                "BELOW_SAFETY" if r["days_cov"] < SAFETY_DAYS else
                "BUILDING"     if r["days_cov"] < TARGET_DAYS else
                "AT_TARGET"
            ),
            "Urgency_Score":  round(urg, 1),
            "Category_Score": cat_s,
            "Indent_Score":   round(ind_s, 1),
            "Final_Score":    round(fs, 2),
        })
    return scores, score_rows

# =============================================================
# SECTION 12 — CHANGEOVER HELPER
# =============================================================

def _co_hrs_for(part, machine, machine_last_part):
    last = machine_last_part.get(machine)
    if last is None or last == part:
        return 0.0
    base = vt_changeover.get(machine, DEFAULT_CHANGEOVER_HRS)
    lc   = part_color.get(last, "UNKNOWN")
    nc   = part_color.get(part, "UNKNOWN")
    purge = (
        COLOR_PURGE_HRS
        if lc != nc and lc != "UNKNOWN" and nc != "UNKNOWN"
        else 0.0
    )
    return base + purge

# =============================================================
# SECTION 13 — MACHINE RANKER
# =============================================================

def rank_machines(part, machines_to_try, machine_hours,
                  machine_last_part, inv_days,
                  machine_cap_override=None):
    """
    machine_cap_override: {machine: available_hours} — used for fixed
    machines where remaining capacity may differ from AVAILABLE_HOURS.
    """
    category  = part_category.get(part, "Stranger")
    fixed_m   = part_fixed_machine.get(part)
    new_color = part_color.get(part, "UNKNOWN")

    # Fixed parts: suppress runner lock (fixed preference overrides it)
    runner_lock = (
        category == "Runner"
        and inv_days <= 1.0
        and fixed_m is None
    )

    # If fixed machine in candidates and has capacity → put it first
    if fixed_m and fixed_m in machines_to_try:
        cap_f  = (machine_cap_override or {}).get(fixed_m, AVAILABLE_HOURS)
        used_f = machine_hours.get(fixed_m, 0)
        free_f = round(cap_f - used_f, 4)
        co_f   = _co_hrs_for(part, fixed_m, machine_last_part)
        eff_f  = round(free_f - co_f, 4)
        if eff_f >= MIN_RUN_HOURS:
            fallback = _rank_normal(
                part,
                [m for m in machines_to_try if m != fixed_m],
                machine_hours, machine_last_part, new_color,
                runner_lock, machine_cap_override,
            )
            return [(fixed_m, co_f, eff_f, -1.0)] + fallback, runner_lock

    return _rank_normal(
        part, machines_to_try, machine_hours, machine_last_part,
        new_color, runner_lock, machine_cap_override,
    ), runner_lock


def _rank_normal(part, machines_to_try, machine_hours, machine_last_part,
                 new_color, runner_lock, machine_cap_override=None):
    ranked = []
    for m in machines_to_try:
        cap  = (machine_cap_override or {}).get(m, AVAILABLE_HOURS)
        used = machine_hours.get(m, 0)
        free = round(cap - used, 4)
        if free < MIN_RUN_HOURS:
            continue
        last = machine_last_part.get(m)
        if runner_lock and last != part:
            continue

        if last is None or last == part:
            co_hrs      = 0.0
            color_bonus = 0.0
        else:
            base       = vt_changeover.get(m, DEFAULT_CHANGEOVER_HRS)
            lc         = part_color.get(last, "UNKNOWN")
            same_color = (lc == new_color and lc != "UNKNOWN" and new_color != "UNKNOWN")
            purge      = (
                0.0 if same_color else (
                    COLOR_PURGE_HRS
                    if lc != "UNKNOWN" and new_color != "UNKNOWN" else 0.0
                )
            )
            co_hrs      = base + purge
            color_bonus = -0.08 if same_color else 0.0

        eff = round(free - co_hrs, 4)
        if eff < MIN_RUN_HOURS:
            continue

        pc   = machine_part_count.get(m, max_part_count)
        cost = (
            pc / max_part_count
            + (co_hrs / AVAILABLE_HOURS) * 0.3
            + (used  / AVAILABLE_HOURS) * 0.2
            + (-0.15 if last == part else 0.0)
            + color_bonus
        )
        ranked.append((m, co_hrs, eff, cost))

    ranked.sort(key=lambda x: x[3])
    return ranked

# =============================================================
# SECTION 14 — MULTI-PART MACHINE PLAN
# =============================================================

def plan_machine_parts(machine, parts_sorted_by_priority, machine_hours,
                        machine_last_part, current_inventory,
                        plan, already_planned, priority_scores,
                        scenario_id, available_cap=None):
    """
    Plans multiple parts on a single machine.
    Order:
      1. Cover daily indent of each part (highest priority first)
         until machine hours exhausted or all indents met.
      2. Extend both/all toward OPD cap with remaining hours,
         highest priority score first, cycling until hours gone.

    available_cap: override for total machine capacity this session
                   (used for non-fixed hours on fixed machines)

    Returns list of new plan rows appended.
    """
    cap         = available_cap if available_cap is not None else AVAILABLE_HOURS
    used        = machine_hours.get(machine, 0)
    remaining   = round(cap - used, 4)
    new_rows    = []

    if remaining < MIN_RUN_HOURS:
        return new_rows

    # ── Phase 1: Cover daily indent of each part ──────────────
    for part in parts_sorted_by_priority:
        if remaining < MIN_RUN_HOURS:
            break
        daily   = indent_daily.get(part, 0)
        r_val   = rate.get(part, 1)
        inv_now = current_inventory.get(part, 0)
        color   = part_color.get(part, "UNKNOWN")
        monthly = indent_monthly.get(part, 0)
        score   = priority_scores.get(part, 0)
        cat     = part_category.get(part, "Stranger")
        fixed_m = part_fixed_machine.get(part)

        shortfall = max(0.0, daily - inv_now)
        hrs_need  = max(MIN_RUN_HOURS, shortfall / r_val if r_val > 0 else MIN_RUN_HOURS)

        co_hrs  = _co_hrs_for(part, machine, machine_last_part)
        eff_cap = round(remaining - co_hrs, 4)
        if eff_cap < MIN_RUN_HOURS:
            continue

        run_hrs = min(eff_cap, hrs_need)
        run_hrs = max(run_hrs, MIN_RUN_HOURS)
        qty     = round(run_hrs * r_val, 0)

        last_m     = machine_last_part.get(machine)
        purge_flag = (
            last_m is not None and last_m != part
            and part_color.get(last_m, "UNKNOWN") != color
            and part_color.get(last_m, "UNKNOWN") != "UNKNOWN"
            and color != "UNKNOWN"
        )
        fixed_used = (fixed_m is not None and machine == fixed_m)

        machine_hours[machine]   = round(machine_hours.get(machine, 0) + co_hrs + run_hrs, 4)
        current_inventory[part]  = round(current_inventory.get(part, 0) + qty, 0)
        machine_last_part[machine] = part
        already_planned.add(part)
        remaining = round(remaining - co_hrs - run_hrs, 4)

        row = {
            "Part":             part,
            "Color":            color,
            "Category":         cat,
            "Fixed_Machine":    fixed_m or "—",
            "Fixed_Used":       "YES" if fixed_used else ("FALLBACK" if fixed_m else "N/A"),
            "Machine":          machine,
            "Run_Hours":        round(run_hrs, 3),
            "Changeover_Hrs":   round(co_hrs, 3),
            "Total_Hrs_Used":   round(co_hrs + run_hrs, 3),
            "Rate_Per_Hour":    round(r_val, 2),
            "Production_Qty":   qty,
            "Monthly_Indent":   round(monthly, 0),
            "Daily_Indent":     round(daily, 2),
            "Today_Target":     round(today_target_qty.get(part, 0), 0),
            "Changeover":       "No" if co_hrs == 0 else "Yes",
            "Color_Purge":      "Yes" if purge_flag else "No",
            "Type":             "Primary" + (" [ZERO-INV]" if inv_now == 0 else ""),
            "Role":             "Primary",
            "Tools_Available":  tools_available.get(part, 1),
            "Tools_Used":       1,
            "Runner_Lock":      "No",
            "Priority_Score":   score,
            "Phase":            1,
            "Indent_Met":       "YES" if qty >= shortfall - 0.5 else "NO — shortfall",
            "Stagger_Adjusted": "No",
        }
        new_rows.append(row)
        plan.append(row)

    # ── Phase 2: Extend all parts toward OPD cap ─────────────
    # Cycle through parts by priority score until hours exhausted
    cap_qtys = {
        p: opd_cap(scenario_id) * indent_daily.get(p, 0)
        for p in parts_sorted_by_priority
    }
    still_extending = True
    while still_extending and remaining >= 0.05:
        still_extending = False
        for part in parts_sorted_by_priority:
            if remaining < 0.05:
                break
            r_val    = rate.get(part, 1)
            inv_now  = current_inventory.get(part, 0)
            headroom = max(0.0, cap_qtys[part] - inv_now)
            if headroom <= 0:
                continue
            ext_hrs  = min(remaining, headroom / r_val if r_val > 0 else 0)
            if ext_hrs < 0.05:
                continue
            extra_qty = round(ext_hrs * r_val, 0)

            # Extend the existing row for this part on this machine
            for row in new_rows:
                if row["Part"] == part and row["Machine"] == machine:
                    row["Run_Hours"]      = round(float(row["Run_Hours"]) + ext_hrs, 3)
                    row["Total_Hrs_Used"] = round(
                        float(row["Changeover_Hrs"]) + float(row["Run_Hours"]), 3
                    )
                    row["Production_Qty"] = round(float(row["Production_Qty"]) + extra_qty, 0)
                    row["Type"]           = str(row["Type"]) + "+InvBuild"
                    break

            machine_hours[machine]  = round(machine_hours.get(machine, 0) + ext_hrs, 4)
            current_inventory[part] = round(current_inventory.get(part, 0) + extra_qty, 0)
            remaining               = round(remaining - ext_hrs, 4)
            still_extending         = True

            print(f"      ↳ INV-BUILD {part:25s} on {machine:15s}  "
                  f"+{ext_hrs:.2f}h  qty+={extra_qty:.0f}")

    return new_rows

# =============================================================
# SECTION 15 — TOOL-AWARE ASSIGNMENT (single part, any machine)
# =============================================================

def assign_part(part, scenario_id, machine_hours, machine_last_part,
                current_inventory, plan, already_planned,
                priority_scores, machine_cap_override=None):
    """
    Standard tool-aware assignment for a single part.
    Fixed machine is tried first via rank_machines().
    Cross-machine spill only if fixed assigned + indent short + tools > 1.
    """
    daily      = indent_daily.get(part, 0)
    monthly    = indent_monthly.get(part, 0)
    r_val      = rate.get(part, 1)
    inv_now    = current_inventory.get(part, 0)
    category   = part_category.get(part, "Stranger")
    tools      = tools_available.get(part, 1)
    score      = priority_scores.get(part, 0)
    color      = part_color.get(part, "UNKNOWN")
    inv_days   = inv_now / daily if daily > 0 else 999
    compatible = compat.get(part, [])
    fixed_m    = part_fixed_machine.get(part)

    if not compatible:
        return []

    total_shortfall = max(0.0, daily - inv_now)
    hrs_for_full    = max(MIN_RUN_HOURS, total_shortfall / r_val if r_val > 0 else MIN_RUN_HOURS)

    new_rows        = []
    produced_so_far = 0.0
    tools_used      = 0
    used_machines   = set()

    ranked, runner_lock = rank_machines(
        part, compatible, machine_hours, machine_last_part,
        inv_days, machine_cap_override
    )
    if not ranked:
        return []

    m1, co1, eff1, _ = ranked[0]
    run1 = max(MIN_RUN_HOURS, min(eff1, hrs_for_full))
    qty1 = round(run1 * r_val, 0)

    machine_hours[m1]       = round(machine_hours.get(m1, 0) + co1 + run1, 4)
    current_inventory[part] = round(current_inventory.get(part, 0) + qty1, 0)
    machine_last_part[m1]   = part
    produced_so_far        += qty1
    tools_used             += 1
    used_machines.add(m1)
    already_planned.add(part)

    fixed_used    = (fixed_m is not None and m1 == fixed_m)
    indent_met    = (produced_so_far >= total_shortfall - 0.5)
    last_m1       = machine_state.get(m1)
    purge_applied = (
        last_m1 is not None and last_m1 != part
        and part_color.get(last_m1, "UNKNOWN") != color
        and part_color.get(last_m1, "UNKNOWN") != "UNKNOWN"
        and color != "UNKNOWN"
    )

    type_tag = "Primary" + (" [ZERO-INV]" if inv_now == 0 else "")
    if fixed_used:
        type_tag += " [FIXED]"
    elif fixed_m:
        type_tag += " [FIXED-FALLBACK]"

    new_rows.append({
        "Part":             part,
        "Color":            color,
        "Category":         category,
        "Fixed_Machine":    fixed_m or "—",
        "Fixed_Used":       "YES" if fixed_used else ("FALLBACK" if fixed_m else "N/A"),
        "Machine":          m1,
        "Run_Hours":        round(run1, 3),
        "Changeover_Hrs":   round(co1, 3),
        "Total_Hrs_Used":   round(co1 + run1, 3),
        "Rate_Per_Hour":    round(r_val, 2),
        "Production_Qty":   qty1,
        "Monthly_Indent":   round(monthly, 0),
        "Daily_Indent":     round(daily, 2),
        "Today_Target":     round(today_target_qty.get(part, 0), 0),
        "Changeover":       "No" if co1 == 0 else "Yes",
        "Color_Purge":      "Yes" if purge_applied else "No",
        "Type":             type_tag,
        "Role":             "Primary",
        "Tools_Available":  tools,
        "Tools_Used":       1,
        "Runner_Lock":      "YES" if runner_lock else "No",
        "Priority_Score":   score,
        "Phase":            1,
        "Indent_Met":       "YES" if indent_met else "NO — shortfall",
        "Stagger_Adjusted": "No",
    })

    # ── Cross-machine spill (Phase 2) ─────────────────────────
    # Only if: fixed assigned first + indent short + tools > 1
    if not indent_met:
        # If part is fixed and we just used its fixed machine, spill allowed
        # If part is fixed but fixed machine had no capacity (fallback used),
        # no spill since we never ran the fixed machine
        fixed_assigned_first = (fixed_m is None) or (m1 == fixed_m)

        if fixed_assigned_first:
            is_critical   = (inv_now == 0)
            tool_hard_cap = tools if is_critical else min(2, tools)

            while produced_so_far < (total_shortfall - 0.5) and tools_used < tool_hard_cap:
                shortfall_now = total_shortfall - produced_so_far
                hrs_needed    = max(
                    MIN_RUN_HOURS,
                    shortfall_now / r_val if r_val > 0 else MIN_RUN_HOURS,
                )
                remaining_m = [m for m in compatible if m not in used_machines]
                ranked_next, _ = rank_machines(
                    part, remaining_m, machine_hours,
                    machine_last_part, inv_days, machine_cap_override
                )
                if not ranked_next:
                    break

                mx, cox, effx, _ = ranked_next[0]
                run_x = max(MIN_RUN_HOURS, min(effx, hrs_needed))
                qty_x = round(run_x * r_val, 0)

                machine_hours[mx]       = round(machine_hours.get(mx, 0) + cox + run_x, 4)
                current_inventory[part] = round(current_inventory.get(part, 0) + qty_x, 0)
                machine_last_part[mx]   = part
                produced_so_far        += qty_x
                tools_used             += 1
                used_machines.add(mx)

                indent_met_x = (produced_so_far >= total_shortfall - 0.5)
                last_mx      = machine_state.get(mx)
                purge_x      = (
                    last_mx is not None and last_mx != part
                    and part_color.get(last_mx, "UNKNOWN") != color
                    and part_color.get(last_mx, "UNKNOWN") != "UNKNOWN"
                    and color != "UNKNOWN"
                )

                new_rows.append({
                    "Part":             part,
                    "Color":            color,
                    "Category":         category,
                    "Fixed_Machine":    fixed_m or "—",
                    "Fixed_Used":       "N/A — expansion",
                    "Machine":          mx,
                    "Run_Hours":        round(run_x, 3),
                    "Changeover_Hrs":   round(cox, 3),
                    "Total_Hrs_Used":   round(cox + run_x, 3),
                    "Rate_Per_Hour":    round(r_val, 2),
                    "Production_Qty":   qty_x,
                    "Monthly_Indent":   round(monthly, 0),
                    "Daily_Indent":     round(daily, 2),
                    "Today_Target":     round(today_target_qty.get(part, 0), 0),
                    "Changeover":       "No" if cox == 0 else "Yes",
                    "Color_Purge":      "Yes" if purge_x else "No",
                    "Type":             "Tool-Expansion",
                    "Role":             f"Tool-Expansion (tool {tools_used}/{tools})",
                    "Tools_Available":  tools,
                    "Tools_Used":       tools_used,
                    "Runner_Lock":      "No",
                    "Priority_Score":   score,
                    "Phase":            2,
                    "Indent_Met":       "YES" if indent_met_x else "NO — shortfall",
                    "Stagger_Adjusted": "No",
                })
                print(f"      ↳ TOOL-EXP {part:26s} tool {tools_used}/{tool_hard_cap} → "
                      f"{mx:15s}  {run_x:.2f}h  qty={qty_x:.0f}  "
                      f"{'COVERED ✓' if indent_met_x else 'still short'}")

    # ── Phase 3: Inventory build ─────────────────────────────
    cap_days     = opd_cap(scenario_id)
    cap_qty      = cap_days * daily
    headroom_qty = max(0.0, cap_qty - current_inventory.get(part, 0))

    if headroom_qty > 0:
        for row in new_rows:
            if headroom_qty <= 0:
                break
            m      = row["Machine"]
            cap_m  = (machine_cap_override or {}).get(m, AVAILABLE_HOURS)
            free_m = round(cap_m - machine_hours.get(m, 0), 4)
            if free_m < 0.05:
                continue
            ext = min(free_m, headroom_qty / r_val if r_val > 0 else 0)
            if ext < 0.05:
                continue
            extra_qty = round(ext * r_val, 0)

            row["Run_Hours"]      = round(float(row["Run_Hours"]) + ext, 3)
            row["Total_Hrs_Used"] = round(
                float(row["Changeover_Hrs"]) + float(row["Run_Hours"]), 3
            )
            row["Production_Qty"] = round(float(row["Production_Qty"]) + extra_qty, 0)
            row["Type"]           = str(row["Type"]) + "+InvBuild"

            machine_hours[m]        = round(machine_hours.get(m, 0) + ext, 4)
            current_inventory[part] = round(current_inventory.get(part, 0) + extra_qty, 0)
            headroom_qty           -= extra_qty
            print(f"      ↳ INV-BUILD {part:25s} on {m:15s}  "
                  f"+{ext:.2f}h  qty+={extra_qty:.0f}")

    for row in new_rows:
        row["Tools_Used"] = tools_used
    return new_rows

# =============================================================
# SECTION 16 — DISPLACEMENT ENGINE
# =============================================================

def _is_displaceable(victim_part, victim_row, current_inventory):
    """
    A plan row is displaceable if:
      - Victim is Stranger (always displaceable), OR
      - Victim is Repeater with inv >= REPEATER_VICTIM_DAYS × daily_indent
    A Runner is never displaced.
    """
    cat   = part_category.get(victim_part, "Stranger")
    if cat == "Runner":
        return False
    if cat == "Stranger":
        return True
    # Repeater — only if has enough stock
    daily  = indent_daily.get(victim_part, 0)
    inv    = current_inventory.get(victim_part, 0)
    days   = inv / daily if daily > 0 else 999
    return days >= REPEATER_VICTIM_DAYS


def displace_and_assign(part, machine_hours, machine_last_part,
                         current_inventory, plan, already_planned,
                         priority_scores, scenario_id):
    """
    Generalized displacement:
    - Eligible displacers: Runner with inv < RUNNER_DISPLACE_DAYS × daily,
      OR Repeater with inv = 0
    - Finds machine where yieldable rows (Strangers + eligible Repeaters)
      can free enough hours for the displacer
    - Victim sort: lowest daily_indent + highest planned qty (least impact)
    - Returns True if placed, False otherwise
    """
    daily      = indent_daily.get(part, 0)
    r_val      = rate.get(part, 1)
    inv_now    = current_inventory.get(part, 0)
    category   = part_category.get(part, "Stranger")
    score      = priority_scores.get(part, 0)
    compatible = compat.get(part, [])
    fixed_m    = part_fixed_machine.get(part)
    color      = part_color.get(part, "UNKNOWN")

    if not compatible:
        return False

    shortfall_qty = max(0.0, daily - inv_now)
    hours_needed  = max(
        MIN_RUN_HOURS,
        shortfall_qty / r_val if r_val > 0 else MIN_RUN_HOURS,
    )

    # Prefer fixed machine first in candidate list
    ordered = compatible[:]
    if fixed_m and fixed_m in ordered:
        ordered = [fixed_m] + [m for m in ordered if m != fixed_m]

    best_machine    = None
    best_co_hrs     = 0.0
    best_victims    = []
    best_disruption = float("inf")

    for m in ordered:
        co_hrs   = _co_hrs_for(part, m, machine_last_part)
        used_hrs = machine_hours.get(m, 0)
        free_hrs = round(AVAILABLE_HOURS - used_hrs, 4)
        eff_free = round(free_hrs - co_hrs, 4)

        # Machine already has enough free capacity
        if eff_free >= hours_needed:
            run_h = max(MIN_RUN_HOURS, min(eff_free, hours_needed))
            qty   = round(run_h * r_val, 0)

            last_m     = machine_last_part.get(m)
            purge      = (
                last_m is not None and last_m != part
                and part_color.get(last_m, "UNKNOWN") != color
                and part_color.get(last_m, "UNKNOWN") != "UNKNOWN"
                and color != "UNKNOWN"
            )
            fixed_used = (fixed_m is not None and m == fixed_m)

            machine_hours[m]          = round(used_hrs + co_hrs + run_h, 4)
            current_inventory[part]   = round(current_inventory.get(part, 0) + qty, 0)
            machine_last_part[m]      = part
            already_planned.add(part)

            plan.append({
                "Part":             part,
                "Color":            color,
                "Category":         category,
                "Fixed_Machine":    fixed_m or "—",
                "Fixed_Used":       "YES" if fixed_used else ("FALLBACK" if fixed_m else "N/A"),
                "Machine":          m,
                "Run_Hours":        round(run_h, 3),
                "Changeover_Hrs":   round(co_hrs, 3),
                "Total_Hrs_Used":   round(co_hrs + run_h, 3),
                "Rate_Per_Hour":    round(r_val, 2),
                "Production_Qty":   qty,
                "Monthly_Indent":   round(indent_monthly.get(part, 0), 0),
                "Daily_Indent":     round(daily, 2),
                "Today_Target":     round(today_target_qty.get(part, 0), 0),
                "Changeover":       "No" if co_hrs == 0 else "Yes",
                "Color_Purge":      "Yes" if purge else "No",
                "Type":             f"Displacement [free capacity] [{category}]",
                "Role":             "Primary",
                "Tools_Available":  tools_available.get(part, 1),
                "Tools_Used":       1,
                "Runner_Lock":      "No",
                "Priority_Score":   score,
                "Phase":            1,
                "Indent_Met":       "YES" if qty >= shortfall_qty - 0.5 else "NO — partial",
                "Stagger_Adjusted": "No",
            })
            print(f"    ↳ DISPLACE (free cap)  {part:28s} → {m:15s}  "
                  f"run={run_h:.2f}h  qty={qty:.0f}  ✓")
            return True

        # Check yieldable rows
        machine_rows = [r for r in plan if r["Machine"] == m]
        yieldable    = [
            r for r in machine_rows
            if _is_displaceable(r["Part"], r, current_inventory)
        ]
        if not yieldable:
            continue

        # Sort victims: lowest daily_indent first, highest planned_qty first
        yieldable.sort(key=lambda r: (
            indent_daily.get(r["Part"], 0),
            -float(r.get("Production_Qty", 0)),
        ))

        reclaim_detail = []
        for row in yieldable:
            rr = float(row.get("Run_Hours", 0))
            if rr <= 0:
                continue
            if rr > MIN_RUN_HOURS:
                reclaim_detail.append((row, round(rr - MIN_RUN_HOURS, 4), "partial"))
            else:
                reclaim_detail.append((row, round(rr, 4), "full_remove"))

        total_reclaim = sum(x[1] for x in reclaim_detail)
        eff_after     = round(free_hrs + total_reclaim - co_hrs, 4)

        if eff_after < MIN_RUN_HOURS or eff_after < hours_needed:
            continue

        disruption = total_reclaim
        if disruption < best_disruption:
            best_disruption = disruption
            best_machine    = m
            best_co_hrs     = co_hrs
            best_victims    = reclaim_detail

    if best_machine is None:
        return False

    # Execute carve
    hours_to_free = hours_needed
    total_freed   = 0.0

    for (victim_row, reclaim_hrs, reclaim_type) in best_victims:
        if hours_to_free <= 0.001:
            break
        vpart      = victim_row["Part"]
        v_rate     = rate.get(vpart, 1)
        v_run_orig = float(victim_row.get("Run_Hours", 0))
        carve      = round(min(reclaim_hrs, hours_to_free), 4)
        if carve <= 0:
            continue

        new_run = round(v_run_orig - carve, 4)
        if new_run < MIN_RUN_HOURS:
            plan.remove(victim_row)
            machine_hours[best_machine] = round(
                machine_hours.get(best_machine, 0) - v_run_orig, 4
            )
            lost_qty = round(v_run_orig * v_rate, 0)
            current_inventory[vpart] = round(
                current_inventory.get(vpart, 0) - lost_qty, 0
            )
            actually_freed = v_run_orig
            print(f"    ↳ YIELD (remove)  {vpart:28s}  freed {v_run_orig:.2f}h  "
                  f"−{lost_qty:.0f} pcs")
        else:
            lost_qty = round(carve * v_rate, 0)
            victim_row["Run_Hours"]      = new_run
            victim_row["Production_Qty"] = round(new_run * v_rate, 0)
            victim_row["Total_Hrs_Used"] = round(
                float(victim_row.get("Changeover_Hrs", 0)) + new_run, 3
            )
            victim_row["Type"] = str(victim_row.get("Type", "")) + " [YIELDED]"
            machine_hours[best_machine] = round(
                machine_hours.get(best_machine, 0) - carve, 4
            )
            current_inventory[vpart] = round(
                current_inventory.get(vpart, 0) - lost_qty, 0
            )
            actually_freed = carve
            print(f"    ↳ YIELD (reduce)  {vpart:28s}  −{carve:.2f}h  "
                  f"−{lost_qty:.0f} pcs  rem={round(new_run * v_rate,0):.0f}")

        hours_to_free = round(hours_to_free - actually_freed, 4)
        total_freed   = round(total_freed + actually_freed, 4)

    # Assign displaced part
    used_now  = machine_hours.get(best_machine, 0)
    free_now  = round(AVAILABLE_HOURS - used_now, 4)
    eff_free  = round(free_now - best_co_hrs, 4)

    if eff_free < MIN_RUN_HOURS:
        return False

    run_hrs    = max(MIN_RUN_HOURS, min(eff_free, hours_needed))
    qty        = round(run_hrs * r_val, 0)
    last_m     = machine_last_part.get(best_machine)
    purge      = (
        last_m is not None and last_m != part
        and part_color.get(last_m, "UNKNOWN") != color
        and part_color.get(last_m, "UNKNOWN") != "UNKNOWN"
        and color != "UNKNOWN"
    )
    fixed_used = (fixed_m is not None and best_machine == fixed_m)

    machine_hours[best_machine]     = round(used_now + best_co_hrs + run_hrs, 4)
    current_inventory[part]         = round(current_inventory.get(part, 0) + qty, 0)
    machine_last_part[best_machine] = part
    already_planned.add(part)

    plan.append({
        "Part":             part,
        "Color":            color,
        "Category":         category,
        "Fixed_Machine":    fixed_m or "—",
        "Fixed_Used":       "YES" if fixed_used else ("FALLBACK" if fixed_m else "N/A"),
        "Machine":          best_machine,
        "Run_Hours":        round(run_hrs, 3),
        "Changeover_Hrs":   round(best_co_hrs, 3),
        "Total_Hrs_Used":   round(best_co_hrs + run_hrs, 3),
        "Rate_Per_Hour":    round(r_val, 2),
        "Production_Qty":   qty,
        "Monthly_Indent":   round(indent_monthly.get(part, 0), 0),
        "Daily_Indent":     round(daily, 2),
        "Today_Target":     round(today_target_qty.get(part, 0), 0),
        "Changeover":       "No" if best_co_hrs == 0 else "Yes",
        "Color_Purge":      "Yes" if purge else "No",
        "Type":             f"Displacement [{category}]",
        "Role":             "Primary",
        "Tools_Available":  tools_available.get(part, 1),
        "Tools_Used":       1,
        "Runner_Lock":      "No",
        "Priority_Score":   score,
        "Phase":            1,
        "Indent_Met":       "YES" if qty >= shortfall_qty - 0.5 else "NO — partial",
        "Stagger_Adjusted": "No",
    })
    print(f"    ✓ DISPLACE {part:32s} → {best_machine}  "
          f"run={run_hrs:.2f}h  qty={qty:.0f}  "
          f"reclaimed={total_freed:.2f}h  purge={'Yes' if purge else 'No'}")
    return True

# =============================================================
# SECTION 17 — TOOL-CHANGER HELPERS
# =============================================================

def _fmt_h(h):
    try:
        total_min = int(round(float(h) * 60))
        return f"{total_min // 60:02d}:{total_min % 60:02d}"
    except Exception:
        return "??"

def _collect_co_events(plan, machines):
    events = []
    for m in machines:
        m_rows = [r for r in plan if r["Machine"] == m]
        if len(m_rows) < 2:
            continue
        cursor = 0.0
        for i, row in enumerate(m_rows):
            co_h  = float(row.get("Changeover_Hrs") or 0.0)
            run_h = float(row.get("Run_Hours")       or 0.0)
            if co_h > 0 and i > 0:
                events.append({
                    "machine":       m,
                    "part_before":   m_rows[i-1]["Part"],
                    "part_after":    row["Part"],
                    "co_duration":   co_h,
                    "natural_start": cursor,
                    "row_before":    m_rows[i-1],
                    "row_after":     row,
                    "actual_start":  None,
                    "wait_hrs":      0.0,
                })
            cursor += co_h + run_h
    return events

def _recompute_natural_start(ev, plan):
    m, target = ev["machine"], ev["row_after"]
    cursor = 0.0
    for r in [r for r in plan if r["Machine"] == m]:
        if r is target:
            break
        cursor += float(r.get("Changeover_Hrs") or 0) + float(r.get("Run_Hours") or 0)
    return cursor

def _machine_spare(m, plan):
    used = sum(
        float(r.get("Changeover_Hrs") or 0) + float(r.get("Run_Hours") or 0)
        for r in plan if r["Machine"] == m
    )
    return max(0.0, AVAILABLE_HOURS - used)

def _extend_row_before(ev, wait_hrs, plan):
    spare     = _machine_spare(ev["machine"], plan)
    extend_by = min(wait_hrs, spare)
    if extend_by <= 0:
        return 0.0, 0
    rb    = ev["row_before"]
    r_val = rate.get(rb["Part"], 1.0)
    extra = round(extend_by * r_val, 0)
    rb["Run_Hours"]      = round(float(rb.get("Run_Hours") or 0) + extend_by, 3)
    rb["Production_Qty"] = round(float(rb.get("Production_Qty") or 0) + extra, 0)
    rb["Total_Hrs_Used"] = round(
        float(rb.get("Changeover_Hrs") or 0) + float(rb["Run_Hours"]), 3
    )
    rb["Stagger_Adjusted"] = f"CO: +{round(extend_by*60,1)}min fill"
    return extend_by, extra

# =============================================================
# SECTION 18 — CO STAGGER
# =============================================================

MIN_CO_GAP_HRS = 20 / 60.0

def _finish_time_of_co(ev, plan):
    m, target = ev["machine"], ev["row_after"]
    total = 0.0
    for row in plan:
        if row["Machine"] != m:
            continue
        if row is target:
            break
        total += float(row.get("Run_Hours") or 0)
        total += float(row.get("Changeover_Hrs") or 0)
    return round(total, 4)

def stagger_co_by_quantity(plan, machines, scenario_id, current_inventory):
    events = _collect_co_events(plan, machines)
    if len(events) < 2:
        return 0
    adjustments = 0
    for _ in range(len(events) * 2):
        for ev in events:
            ev["_ft"] = _finish_time_of_co(ev, plan)
        events.sort(key=lambda e: e["_ft"])
        conflict = None
        for i in range(len(events) - 1):
            req = events[i]["co_duration"] + MIN_CO_GAP_HRS
            gap = events[i+1]["_ft"] - events[i]["_ft"]
            if gap < req - 0.001:
                conflict = (events[i], events[i+1], gap, req)
                break
        if conflict is None:
            break
        ev_e, ev_l, gap, req = conflict
        shortfall = req - gap

        row_l  = ev_l["row_before"]
        p_l    = row_l["Part"]
        r_l    = rate.get(p_l, 1)
        inv_l  = current_inventory.get(p_l, 0)
        cap_q  = opd_cap(scenario_id) * indent_daily.get(p_l, 0)
        used_m = sum(
            float(r.get("Changeover_Hrs") or 0) + float(r.get("Run_Hours") or 0)
            for r in plan if r["Machine"] == ev_l["machine"]
        )
        free_m = max(0.0, AVAILABLE_HOURS - used_m)
        push_q = round(shortfall * r_l, 0)
        headroom = max(0.0, cap_q - inv_l)

        if push_q <= headroom and shortfall <= free_m + 0.001 and r_l > 0:
            row_l["Run_Hours"]      = round(float(row_l.get("Run_Hours") or 0) + shortfall, 3)
            row_l["Production_Qty"] = round(float(row_l.get("Production_Qty") or 0) + push_q, 0)
            row_l["Total_Hrs_Used"] = round(
                float(row_l.get("Changeover_Hrs") or 0) + float(row_l["Run_Hours"]), 3
            )
            row_l["Stagger_Adjusted"] = f"CO-stagger PUSH +{round(shortfall*60,1)}min"
            current_inventory[p_l] = round(current_inventory.get(p_l, 0) + push_q, 0)
            adjustments += 1
            continue

        row_e  = ev_e["row_before"]
        p_e    = row_e["Part"]
        r_e    = rate.get(p_e, 1)
        prod_e = float(row_e.get("Production_Qty") or 0)
        daily_e = indent_daily.get(p_e, 0)
        inv_e  = current_inventory.get(p_e, 0)
        min_q  = max(0.0, daily_e - inv_e)
        pull_q = round(shortfall * r_e, 0)
        max_pull = max(0.0, prod_e - min_q)

        if pull_q <= max_pull and r_e > 0 and prod_e - pull_q >= MIN_RUN_HOURS * r_e:
            row_e["Run_Hours"]      = round(float(row_e.get("Run_Hours") or 0) - shortfall, 3)
            row_e["Production_Qty"] = round(prod_e - pull_q, 0)
            row_e["Total_Hrs_Used"] = round(
                float(row_e.get("Changeover_Hrs") or 0) + float(row_e["Run_Hours"]), 3
            )
            row_e["Stagger_Adjusted"] = f"CO-stagger PULL -{round(shortfall*60,1)}min"
            current_inventory[p_e] = round(current_inventory.get(p_e, 0) - pull_q, 0)
            adjustments += 1
            continue

        ev_l["_ft"] = ev_e["_ft"] + MIN_CO_GAP_HRS
        break
    return adjustments

def stagger_changeovers_serial_queue(plan, machines):
    print(f"\n  Tool-Changer Serial Queue")
    events = _collect_co_events(plan, machines)
    if not events:
        print(f"  No changeovers — tool changer idle  ✓")
        return
    events.sort(key=lambda e: e["natural_start"])
    print(f"  {len(events)} CO events across {len({e['machine'] for e in events})} machines")
    print(f"\n  {'#':<4} {'Machine':<18} {'Before':<22} {'After':<22} "
          f"{'Dur':>5} {'Natural':>8} {'Actual':>8} {'Wait':>7} {'Fill':>6}")
    print(f"  {'─'*100}")
    tc_free = 0.0
    total_extra = 0
    for idx, ev in enumerate(events, 1):
        ns   = _recompute_natural_start(ev, plan)
        co_h = ev["co_duration"]
        as_  = max(ns, tc_free)
        wait = round(as_ - ns, 4)
        tc_free = as_ + co_h
        extra = 0
        if wait > 0.001:
            _, extra = _extend_row_before(ev, wait, plan)
            total_extra += extra
        ev["actual_start"] = as_
        ev["wait_hrs"]     = wait
        print(f"  {idx:<4} {ev['machine']:<18} {ev['part_before']:<22} "
              f"{ev['part_after']:<22} {round(co_h*60,1):>4.0f}m "
              f"{_fmt_h(ns):>8} {_fmt_h(as_):>8} "
              f"{('+'+str(round(wait*60,1))+'m') if wait>0.001 else 'none':>7} "
              f"{('+'+str(int(extra))) if extra>0 else '—':>6}")
    n_w = sum(1 for e in events if e.get("wait_hrs", 0) > 0.001)
    print(f"\n  TC free at {_fmt_h(tc_free)}  |  {n_w}/{len(events)} waited  "
          f"|  +{total_extra:,.0f} extra pcs  |  ZERO OVERLAP guaranteed")

# =============================================================
# SECTION 19 — UTILISATION ENFORCER
# =============================================================

def _add_part_to_machine(p, m, run_hrs, co_hrs, machine_hours,
                          machine_last_part, current_inventory,
                          already_planned, plan, priority_scores,
                          type_label, role_label):
    r_val  = rate.get(p, 1)
    qty    = round(run_hrs * r_val, 0)
    last   = machine_last_part.get(m)
    pc     = part_color.get(p, "UNKNOWN")
    lc     = part_color.get(last, "UNKNOWN") if last else "UNKNOWN"
    purge  = last is not None and last != p and pc != lc and pc != "UNKNOWN" and lc != "UNKNOWN"

    machine_hours[m]     = round(machine_hours.get(m, 0) + co_hrs + run_hrs, 4)
    current_inventory[p] = round(current_inventory.get(p, 0) + qty, 0)
    machine_last_part[m] = p
    already_planned.add(p)

    plan.append({
        "Part":             p,
        "Color":            pc,
        "Category":         part_category.get(p, "Stranger"),
        "Fixed_Machine":    part_fixed_machine.get(p, "—"),
        "Fixed_Used":       "N/A — enforcer",
        "Machine":          m,
        "Run_Hours":        round(run_hrs, 3),
        "Changeover_Hrs":   round(co_hrs, 3),
        "Total_Hrs_Used":   round(co_hrs + run_hrs, 3),
        "Rate_Per_Hour":    round(r_val, 2),
        "Production_Qty":   qty,
        "Monthly_Indent":   round(indent_monthly.get(p, 0), 0),
        "Daily_Indent":     round(indent_daily.get(p, 0), 2),
        "Today_Target":     round(today_target_qty.get(p, 0), 0),
        "Changeover":       "No" if co_hrs == 0 else "Yes",
        "Color_Purge":      "Yes" if purge else "No",
        "Type":             type_label,
        "Role":             role_label,
        "Tools_Available":  tools_available.get(p, 1),
        "Tools_Used":       1,
        "Runner_Lock":      "No",
        "Priority_Score":   round(priority_scores.get(p, 0), 2),
        "Phase":            1,
        "Stagger_Adjusted": "No",
    })
    return qty


def utilization_enforcer(plan, machine_hours, machine_last_part,
                          all_parts, already_planned,
                          current_inventory, scenario_id,
                          priority_scores, fixed_machines_today):
    """
    fixed_machines_today: set of machines currently in Phase A
                          (skip these in the enforcer)
    """
    print(f"\n  22H UTILIZATION ENFORCER  (floor = {UTIL_TARGET_PCT}%)")
    micro_idle_log = []
    floor_hrs      = AVAILABLE_HOURS * (UTIL_TARGET_PCT / 100.0)

    all_skipped = [
        p for p in all_parts
        if should_skip(p)[0] and rate.get(p, 0) > 0 and indent_monthly.get(p, 0) > 0
    ]

    for m in sorted(all_machines, key=lambda m: machine_hours.get(m, 0)):
        # Skip fixed machines in Phase A — they are fully dedicated
        if m in fixed_machines_today:
            continue

        remaining = round(AVAILABLE_HOURS - machine_hours.get(m, 0), 4)
        if remaining < 0.05:
            continue

        # STEP 0: 90% floor — uncapped extension
        if machine_hours.get(m, 0) < floor_hrs:
            needed    = round(floor_hrs - machine_hours.get(m, 0), 4)
            parts_on_m = sorted(
                [row for row in plan if row["Machine"] == m],
                key=lambda r: priority_scores.get(r["Part"], 0), reverse=True
            )
            for row in parts_on_m:
                if needed <= 0.001:
                    break
                p_ext = row["Part"]
                r_ext = rate.get(p_ext, 1)
                ext   = min(needed, remaining)
                if ext < 0.001:
                    continue
                extra = round(ext * r_ext, 0)
                row["Run_Hours"]      = round(float(row.get("Run_Hours", 0)) + ext, 3)
                row["Production_Qty"] = round(float(row.get("Production_Qty", 0)) + extra, 0)
                row["Total_Hrs_Used"] = round(
                    float(row.get("Changeover_Hrs", 0)) + float(row["Run_Hours"]), 3
                )
                row["Type"] = str(row.get("Type", "")) + "+Floor90"
                machine_hours[m]          = round(machine_hours.get(m, 0) + ext, 4)
                current_inventory[p_ext]  = round(current_inventory.get(p_ext, 0) + extra, 0)
                remaining = round(remaining - ext, 4)
                needed    = round(needed - ext, 4)
                print(f"    [S0-FLOOR90] {p_ext:28s} on {m:15s}  "
                      f"+{ext:.2f}h  qty+={extra:.0f}  "
                      f"util={round(machine_hours.get(m,0)/AVAILABLE_HOURS*100,1)}%")

        if remaining < 0.05:
            continue

        # STEP 1: Extend existing parts (OPD-capped)
        for p in sorted(
            {row["Part"] for row in plan if row["Machine"] == m},
            key=lambda x: priority_scores.get(x, 0), reverse=True
        ):
            if remaining < 0.05:
                break
            daily_p  = indent_daily.get(p, 0)
            r_val    = rate.get(p, 1)
            inv_now  = current_inventory.get(p, 0)
            headroom = max(0.0, opd_cap(scenario_id) * daily_p - inv_now)
            ext      = min(remaining, headroom / r_val if r_val > 0 else 0)
            if ext < 0.05:
                continue
            extra = round(ext * r_val, 0)
            for row in plan:
                if row["Part"] == p and row["Machine"] == m:
                    row["Run_Hours"]      = round(float(row.get("Run_Hours", 0)) + ext, 3)
                    row["Production_Qty"] = round(float(row.get("Production_Qty", 0)) + extra, 0)
                    row["Total_Hrs_Used"] = round(
                        float(row.get("Changeover_Hrs", 0)) + float(row["Run_Hours"]), 3
                    )
                    row["Type"] = str(row.get("Type", "")) + "+Extended"
                    break
            machine_hours[m]     = round(machine_hours.get(m, 0) + ext, 4)
            current_inventory[p] = round(current_inventory.get(p, 0) + extra, 0)
            remaining            = round(remaining - ext, 4)
            print(f"    [S1-EXTEND]  {p:28s} on {m:15s}  +{ext:.2f}h  qty+={extra:.0f}")

        if remaining < MIN_RUN_HOURS:
            continue

        last_on_m  = machine_last_part.get(m)
        last_color = part_color.get(last_on_m, "UNKNOWN") if last_on_m else "UNKNOWN"

        # STEP 2: Unplanned compatible parts
        unplanned = [
            p for p in all_parts
            if p not in already_planned
            and m in compat.get(p, [])
            and not should_skip(p)[0]
            and indent_monthly.get(p, 0) > 0
            and rate.get(p, 0) > 0
            and current_inventory.get(p, 0) < opd_cap(scenario_id) * indent_daily.get(p, 0)
        ]
        unplanned.sort(key=lambda p: (
            0 if (last_on_m is None or last_on_m == p) else 1,
            0 if (part_color.get(p,"UNKNOWN") == last_color and last_color != "UNKNOWN") else 1,
            1 if current_inventory.get(p, 0) == 0 else 0,
            -priority_scores.get(p, 0),
        ))

        for p in unplanned:
            if remaining < MIN_RUN_HOURS:
                break
            co_hrs   = _co_hrs_for(p, m, machine_last_part)
            eff_free = round(remaining - co_hrs, 4)
            if eff_free < MIN_RUN_HOURS:
                continue
            daily_p  = indent_daily.get(p, 0)
            r_val    = rate.get(p, 1)
            inv_now  = current_inventory.get(p, 0)
            headroom = max(0.0, opd_cap(scenario_id) * daily_p - inv_now)
            if headroom <= 0:
                continue
            shortfall = max(0.0, daily_p - inv_now)
            run_hrs   = max(
                MIN_RUN_HOURS,
                min(eff_free, max(shortfall / r_val if r_val > 0 else MIN_RUN_HOURS,
                                  headroom / r_val if r_val > 0 else eff_free))
            )
            qty = _add_part_to_machine(
                p, m, run_hrs, co_hrs, machine_hours, machine_last_part,
                current_inventory, already_planned, plan, priority_scores,
                "Filler-Unplanned", "Primary"
            )
            remaining = round(remaining - co_hrs - run_hrs, 4)
            print(f"    [S2-UNPLAN]  {p:28s} → {m:15s}  {run_hrs:.2f}h  qty={qty:.0f}")

        if remaining < MIN_RUN_HOURS:
            continue

        # STEP 3: Re-run planned parts (spare tool)
        spare_tool_parts = [
            p for p in already_planned
            if m in compat.get(p, [])
            and m not in [r["Machine"] for r in plan if r["Part"] == p]
            and tools_available.get(p, 1) > len({r["Machine"] for r in plan if r["Part"] == p})
        ]
        spare_tool_parts.sort(key=lambda p: (
            0 if (last_on_m is None or last_on_m == p) else 1,
            0 if (part_color.get(p,"UNKNOWN") == last_color and last_color != "UNKNOWN") else 1,
            1 if current_inventory.get(p, 0) == 0 else 0,
            -priority_scores.get(p, 0),
        ))
        for p in spare_tool_parts:
            if remaining < MIN_RUN_HOURS:
                break
            co_hrs   = _co_hrs_for(p, m, machine_last_part)
            eff_free = round(remaining - co_hrs, 4)
            if eff_free < MIN_RUN_HOURS:
                continue
            daily_p  = indent_daily.get(p, 0)
            r_val    = rate.get(p, 1)
            inv_now  = current_inventory.get(p, 0)
            headroom = max(0.0, opd_cap(scenario_id) * daily_p - inv_now)
            if headroom <= 0:
                continue
            shortfall = max(0.0, daily_p - inv_now)
            run_hrs   = max(
                MIN_RUN_HOURS,
                min(eff_free, max(shortfall / r_val if r_val > 0 else MIN_RUN_HOURS,
                                  headroom / r_val if r_val > 0 else eff_free))
            )
            qty = round(run_hrs * r_val, 0)
            machine_hours[m]     = round(machine_hours.get(m, 0) + co_hrs + run_hrs, 4)
            current_inventory[p] = round(current_inventory.get(p, 0) + qty, 0)
            machine_last_part[m] = p
            remaining            = round(remaining - co_hrs - run_hrs, 4)
            tun = len({r["Machine"] for r in plan if r["Part"] == p}) + 1
            pc  = part_color.get(p, "UNKNOWN")
            lp  = machine_state.get(m)
            lc2 = part_color.get(lp, "UNKNOWN") if lp else "UNKNOWN"
            purge_s3 = (lp is not None and lp != p and pc != lc2
                        and pc != "UNKNOWN" and lc2 != "UNKNOWN")
            plan.append({
                "Part": p, "Color": pc,
                "Category": part_category.get(p, "Stranger"),
                "Fixed_Machine": part_fixed_machine.get(p, "—"),
                "Fixed_Used": "N/A — enforcer",
                "Machine": m,
                "Run_Hours": round(run_hrs, 3),
                "Changeover_Hrs": round(co_hrs, 3),
                "Total_Hrs_Used": round(co_hrs + run_hrs, 3),
                "Rate_Per_Hour": round(rate.get(p, 1), 2),
                "Production_Qty": qty,
                "Monthly_Indent": round(indent_monthly.get(p, 0), 0),
                "Daily_Indent": round(daily_p, 2),
                "Today_Target": round(today_target_qty.get(p, 0), 0),
                "Changeover": "No" if co_hrs == 0 else "Yes",
                "Color_Purge": "Yes" if purge_s3 else "No",
                "Type": "Re-run (spare tool)",
                "Role": f"Tool-Expansion (tool {tun})",
                "Tools_Available": tools_available.get(p, 1),
                "Tools_Used": tun,
                "Runner_Lock": "No",
                "Priority_Score": round(priority_scores.get(p, 0), 2),
                "Phase": 3, "Stagger_Adjusted": "No",
            })
            print(f"    [S3-RERUN]   {p:28s} → {m:15s}  {run_hrs:.2f}h  "
                  f"qty={qty:.0f}  tool {tun}/{tools_available.get(p,1)}")
            break

        if remaining < MIN_RUN_HOURS:
            continue

        # STEP 4: Skipped parts
        skipped_cands = sorted(
            [p for p in all_skipped if m in compat.get(p, []) and rate.get(p, 0) > 0],
            key=lambda p: (
                0 if (last_on_m is None or last_on_m == p) else 1,
                0 if (part_color.get(p,"UNKNOWN") == last_color and last_color != "UNKNOWN") else 1,
                -indent_daily.get(p, 0),
            )
        )
        for p in skipped_cands:
            if remaining < MIN_RUN_HOURS:
                break
            co_hrs   = _co_hrs_for(p, m, machine_last_part)
            eff_free = round(remaining - co_hrs, 4)
            if eff_free < MIN_RUN_HOURS:
                continue
            r_val   = rate.get(p, 1)
            run_hrs = max(MIN_RUN_HOURS, min(eff_free,
                indent_monthly.get(p, 0) / r_val if r_val > 0 else eff_free))
            qty = _add_part_to_machine(
                p, m, run_hrs, co_hrs, machine_hours, machine_last_part,
                current_inventory, already_planned, plan, priority_scores,
                "Filler-Skipped", "Primary"
            )
            remaining = round(remaining - co_hrs - run_hrs, 4)
            print(f"    [S4-SKIPPED] {p:28s} → {m:15s}  {run_hrs:.2f}h  qty={qty:.0f}")
            break

        # STEP 5: Log micro-idle
        final_rem = round(AVAILABLE_HOURS - machine_hours.get(m, 0), 4)
        if final_rem >= 0.25:
            util_f = round((1 - final_rem / AVAILABLE_HOURS) * 100, 1)
            if util_f < UTIL_TARGET_PCT:
                micro_idle_log.append({
                    "Machine": m,
                    "Idle_Hrs": round(final_rem, 3),
                    "Utilization_Pct": util_f,
                    "Note": "All options exhausted",
                })
                print(f"    [⚠ IDLE]     {m:15s}  {final_rem:.2f}h idle ({util_f}%)")

    return micro_idle_log

# =============================================================
# SECTION 20 — OUTPUT BUILDERS
# =============================================================

def build_multi_machine_view(plan):
    if not plan:
        return pd.DataFrame()
    from collections import defaultdict
    part_rows = defaultdict(list)
    for row in plan:
        part_rows[row["Part"]].append(row)
    multi = {p: rows for p, rows in part_rows.items() if len(rows) > 1}
    if not multi:
        return pd.DataFrame()
    out = []
    for part, rows in sorted(multi.items(), key=lambda x: -sum(r["Production_Qty"] for r in x[1])):
        daily     = indent_daily.get(part, 0)
        total_qty = sum(float(r["Production_Qty"]) for r in rows)
        for row in rows:
            out.append({
                "Part":                   part,
                "Color":                  part_color.get(part, "UNKNOWN"),
                "Category":               part_category.get(part, "Stranger"),
                "Fixed_Machine":          part_fixed_machine.get(part, "—"),
                "Tools_Available":        tools_available.get(part, 1),
                "Machines_Used":          len(rows),
                "Machine":                row["Machine"],
                "Role":                   row.get("Role", "Primary"),
                "Run_Hours":              round(float(row["Run_Hours"]), 2),
                "Changeover_Hrs":         round(float(row.get("Changeover_Hrs", 0)), 2),
                "Color_Purge":            row.get("Color_Purge", "No"),
                "Production_Qty":         round(float(row["Production_Qty"]), 0),
                "Daily_Indent":           round(daily, 2),
                "Total_Qty_All_Machines": round(total_qty, 0),
                "Type":                   row.get("Type", "—"),
            })
        out.append({
            "Part": f"  ↳ TOTAL — {part}",
            "Color": part_color.get(part, "UNKNOWN"),
            "Category": "—",
            "Fixed_Machine": part_fixed_machine.get(part, "—"),
            "Tools_Available": tools_available.get(part, 1),
            "Machines_Used": len(rows),
            "Machine": f"{len(rows)} machines",
            "Role": "TOTAL",
            "Run_Hours": round(sum(float(r["Run_Hours"]) for r in rows), 2),
            "Changeover_Hrs": round(sum(float(r.get("Changeover_Hrs", 0)) for r in rows), 2),
            "Color_Purge": "—",
            "Production_Qty": round(total_qty, 0),
            "Daily_Indent": round(daily, 2),
            "Total_Qty_All_Machines": round(total_qty, 0),
            "Type": "—",
        })
        out.append({k: "" for k in out[-1].keys()})
    return pd.DataFrame(out)


def build_production_vs_indent(plan, all_parts):
    if not plan:
        return pd.DataFrame()
    from collections import defaultdict
    part_qty      = defaultdict(float)
    part_machines = defaultdict(list)
    for row in plan:
        p = row["Part"]
        part_qty[p]      += float(row.get("Production_Qty", 0))
        part_machines[p].append(row["Machine"])
    rows = []
    for p in sorted(part_qty.keys()):
        daily    = indent_daily.get(p, 0)
        inv_b    = inventory.get(p, 0)
        produced = round(part_qty[p], 0)
        inv_after = round(inv_b + produced, 0)
        gap       = round(produced - daily, 0)
        gap_dir   = "OVER" if gap > 0 else ("UNDER" if gap < 0 else "MET")
        rows.append({
            "Part":               p,
            "Color":              part_color.get(p, "UNKNOWN"),
            "Category":           part_category.get(p, "Stranger"),
            "Fixed_Machine":      part_fixed_machine.get(p, "—"),
            "Machines":           ", ".join(dict.fromkeys(part_machines[p])),
            "Total_Qty_Produced": produced,
            "Daily_Indent":       round(daily, 2),
            "Monthly_Indent":     round(indent_monthly.get(p, 0), 0),
            "Gap_vs_Daily":       gap,
            "Gap_Direction":      gap_dir,
            "Extra_Days_Stock":   round(gap / daily, 2) if daily > 0 and gap > 0 else 0.0,
            "Inventory_Before":   round(inv_b, 0),
            "Inventory_After":    inv_after,
            "Days_Coverage_After":round(inv_after / daily, 2) if daily > 0 else 0,
        })
    df = pd.DataFrame(rows)
    if not df.empty:
        df["_s"] = df["Gap_Direction"].map({"UNDER": 0, "MET": 1, "OVER": 2})
        df = df.sort_values(["_s", "Gap_vs_Daily"]).drop(columns=["_s"]).reset_index(drop=True)
    return df


def build_inventory_target_sheet(plan, all_parts, scenario_id):
    from collections import defaultdict
    part_produced = defaultdict(float)
    for row in plan:
        part_produced[row["Part"]] += float(row.get("Production_Qty", 0))
    rows = []
    for p in sorted(all_parts):
        daily     = indent_daily.get(p, 0)
        inv_b     = inventory.get(p, 0)
        produced  = round(part_produced.get(p, 0), 0)
        inv_after = round(inv_b + produced, 0)
        days_b    = round(inv_b    / daily, 2) if daily > 0 else 0
        days_a    = round(inv_after / daily, 2) if daily > 0 else 0
        tgt_qty   = round(TARGET_DAYS * daily, 0)
        gap_qty   = round(tgt_qty - inv_after, 0)
        gap_days  = max(0, round(gap_qty / daily, 2) if daily > 0 else 0)
        cap       = opd_cap(scenario_id)
        net_gain  = (cap - 1) * daily if daily > 0 else 0
        est_days  = (
            "AT TARGET" if gap_qty <= 0
            else "N/A" if net_gain <= 0
            else str(math.ceil(gap_qty / net_gain)) + " days"
        )
        if inv_after == 0:       status = "CRITICAL"
        elif days_a < SAFETY_DAYS: status = "BELOW_SAFETY"
        elif days_a < TARGET_DAYS: status = "BUILDING"
        else:                       status = "AT_TARGET"
        skip, skip_rsn = should_skip(p)
        rows.append({
            "Part":                   p,
            "Color":                  part_color.get(p, "UNKNOWN"),
            "Category":               part_category.get(p, "Stranger"),
            "Fixed_Machine":          part_fixed_machine.get(p, "—"),
            "Tools":                  tools_available.get(p, 1),
            "Monthly_Indent":         round(indent_monthly.get(p, 0), 0),
            "Daily_Indent":           round(daily, 2),
            "Target_Qty_5days":       tgt_qty,
            "Safety_Floor_Qty_3days": round(SAFETY_DAYS * daily, 0),
            "Inv_Before":             round(inv_b, 0),
            "Days_Before":            days_b,
            "Produced_Today":         produced,
            "Inv_After":              inv_after,
            "Days_After":             days_a,
            "Gap_to_Target_Qty":      max(0, gap_qty),
            "Gap_to_Target_Days":     gap_days,
            "Buffer_Status":          status,
            "OPD_Cap_Today":          cap,
            "Est_Days_to_Target":     est_days,
            "Scheduled_Today":        (
                "YES" if produced > 0
                else "SKIPPED — AT TARGET" if inv_b >= tgt_qty
                else "NO — no capacity"
            ),
            "Skip_Reason":            skip_rsn if skip else "",
        })
    status_order = {"CRITICAL": 0, "BELOW_SAFETY": 1, "BUILDING": 2, "AT_TARGET": 3}
    df = pd.DataFrame(rows)
    if not df.empty:
        df["_s"] = df["Buffer_Status"].map(status_order)
        df = df.sort_values(["_s", "Gap_to_Target_Days"], ascending=[True, False])
        df = df.drop(columns=["_s"]).reset_index(drop=True)
    return df


def build_terminal_status_sheet():
    rows = []
    all_t = set(terminal_status.keys())
    for terms in part_terminals.values():
        all_t.update(terms)
    for t in sorted(all_t):
        inv_t = terminal_status.get(t, None)
        avail = (inv_t is not None and inv_t > 0)
        needing = [p for p, terms in part_terminals.items() if t in terms]
        blocked = [p for p in needing if not avail]
        rows.append({
            "Terminal":              t,
            "Inventory":             round(inv_t, 0) if inv_t is not None else "NOT IN SHEET",
            "Available_Today":       "YES" if avail else "NO — ZERO STOCK",
            "Parts_Requiring_Count": len(needing),
            "Parts_Requiring":       ", ".join(sorted(needing)) if needing else "—",
            "Parts_Blocked_Count":   len(blocked),
            "Parts_Blocked":         ", ".join(sorted(blocked)) if blocked else "—",
            "Impact":                (
                "BLOCKING — reschedule" if blocked
                else "No impact today" if needing
                else "No parts use this terminal"
            ),
        })
    return pd.DataFrame(rows)


def build_fixed_adherence_sheet(plan):
    rows = []
    for part, fixed_m in sorted(part_fixed_machine.items()):
        part_rows = [r for r in plan if r["Part"] == part]
        if not part_rows:
            status = "NOT PLANNED TODAY"
            machine_used = "—"
        else:
            ms = {r["Machine"] for r in part_rows}
            machine_used = ", ".join(sorted(ms))
            if fixed_m in ms and len(ms) == 1:
                status = "FIXED MACHINE USED"
            elif fixed_m in ms:
                status = "FIXED + FALLBACK USED"
            else:
                status = "FALLBACK ONLY — fixed had no capacity"
        daily = indent_daily.get(part, 0)
        r_val = rate.get(part, 1)
        hrs_needed = round(daily / r_val, 2) if r_val > 0 else 0
        phase, _ = fixed_machine_phase(part, {})   # just for display
        rows.append({
            "Part":             part,
            "Category":         part_category.get(part, "Stranger"),
            "Color":            part_color.get(part, "UNKNOWN"),
            "Fixed_Machine":    fixed_m,
            "Fixed_In_Matrix":  "YES" if fixed_m in compat.get(part, []) else "NO",
            "Machine_Used":     machine_used,
            "Status":           status,
            "Daily_Indent":     round(daily, 2),
            "Hrs_To_Cover_Daily": hrs_needed,
            "Uses_23h_Rule":    "YES" if hrs_needed > EXTENDED_THRESHOLD else "No",
            "Inventory":        round(inventory.get(part, 0), 0),
            "Days_Coverage":    round(inventory.get(part, 0) / daily, 2) if daily > 0 else 0,
        })
    return pd.DataFrame(rows)


def compute_indent_horizon(parts):
    rows = []
    for p in parts:
        inv    = inventory.get(p, 0.0)
        monthly = indent_monthly.get(p, 0.0)
        daily  = indent_daily.get(p, 0.0)
        r      = rate.get(p, 1.0)
        skip, skip_rsn = should_skip(p)
        indent_hrs = monthly / r if r > 0 else 0.0
        days_cov   = inv / daily if daily > 0 else 0
        if inv == 0.0 and monthly > 0:   status = "ZERO INV — FORCED"
        elif skip and inv >= TARGET_DAYS * daily: status = "AT TARGET — SKIP"
        elif skip:                        status = "SKIPPED"
        elif today_target_qty.get(p, 0) > 0: status = "PRODUCTION NEEDED"
        elif monthly == 0:                status = "NO INDENT"
        else:                             status = "INV SUFFICIENT"
        rows.append({
            "Part":             p,
            "Color":            part_color.get(p, "UNKNOWN"),
            "Fixed_Machine":    part_fixed_machine.get(p, "—"),
            "Tools":            tools_available.get(p, 1),
            "Monthly_Indent":   round(monthly, 0),
            "Indent_Hrs_Total": round(indent_hrs, 2),
            "Working_Days":     WORKING_DAYS,
            "Daily_Indent":     round(daily, 2),
            "Hrs_For_Daily":    round(daily / r, 2) if r > 0 else 0,
            "Uses_23h_Rule":    "YES" if (daily / r if r > 0 else 0) > EXTENDED_THRESHOLD else "No",
            "Inventory_Now":    round(inv, 0),
            "Days_Coverage":    round(days_cov, 2),
            "Safety_Floor":     SAFETY_DAYS,
            "Target_Ceiling":   TARGET_DAYS,
            "Today_Target_Qty": round(today_target_qty.get(p, 0), 0),
            "Rate_Per_Hour":    round(r, 2),
            "Indent_Status":    status,
            "Skip_Reason":      skip_rsn,
        })
    return pd.DataFrame(rows)

# =============================================================
# SECTION 21 — MAIN SCHEDULER
# =============================================================

def schedule(parts, label=""):
    print(f"\n{'─'*65}")
    print(f"  {label}  |  {len(parts)} parts  |  {len(all_machines)} machines")
    print(f"{'─'*65}")

    scenario_id, scenario_desc = classify_scenario(parts)
    cap_today = opd_cap(scenario_id)
    print(f"  {scenario_desc}  |  OPD cap: {cap_today} days")

    horizon_df = compute_indent_horizon(parts)

    active_parts = [
        p for p in parts
        if not should_skip(p)[0] and indent_monthly.get(p, 0) > 0
    ]

    priority_scores, score_rows = compute_priority_scores(active_parts)
    score_df = pd.DataFrame(score_rows) if score_rows else pd.DataFrame()

    machine_hours     = {m: 0.0 for m in all_machines}
    machine_last_part = {m: machine_state.get(m) for m in all_machines}
    current_inventory = inventory.copy()
    plan              = []
    already_planned   = set()
    not_planned       = []
    deferred          = []
    disp_log          = []   # displacement log rows

    # ── PASS 0: Fixed machine scheduling ─────────────────────
    # Process every fixed machine first, before any other part
    # is assigned. This ensures fixed machines are not pre-empted
    # by normal scheduling.
    fixed_machines_phase_a = set()   # machines locked in Phase A today

    print(f"\n  FIXED MACHINE PASS ({len(machine_fixed_parts)} fixed machines)")

    for fixed_machine, fixed_parts_list in sorted(machine_fixed_parts.items()):
        # Determine phase using start-of-day inventory
        phase, chosen_part = fixed_machine_phase(fixed_machine, current_inventory)

        # Filter to active parts only
        active_fixed = [
            p for p in fixed_parts_list
            if not should_skip(p)[0] and indent_monthly.get(p, 0) > 0
        ]
        if not active_fixed:
            print(f"  {fixed_machine:<25} — no active fixed parts today  (skip)")
            continue

        print(f"\n  {fixed_machine:<25} Phase {phase}  |  "
              f"fixed parts: {', '.join(active_fixed)}")

        if phase == "A":
            # Dedicated: run only the chosen part, full hours
            fixed_machines_phase_a.add(fixed_machine)
            part        = chosen_part
            daily       = indent_daily.get(part, 0)
            r_val       = rate.get(part, 1)
            color       = part_color.get(part, "UNKNOWN")
            monthly     = indent_monthly.get(part, 0)
            score       = priority_scores.get(part, 0)
            cat         = part_category.get(part, "Stranger")

            run_hrs_fixed, _ = fixed_part_hours(part)
            # In Phase A use full capacity for the fixed part (no sharing)
            run_hrs = min(run_hrs_fixed, EXTENDED_HOURS if daily / r_val > EXTENDED_THRESHOLD else AVAILABLE_HOURS)
            # Extend to fill all AVAILABLE_HOURS (machine should be fully utilised)
            run_hrs = max(run_hrs, AVAILABLE_HOURS)
            run_hrs = min(run_hrs, EXTENDED_HOURS if daily / r_val > EXTENDED_THRESHOLD else AVAILABLE_HOURS)
            qty     = round(run_hrs * r_val, 0)

            co_hrs  = _co_hrs_for(part, fixed_machine, machine_last_part)
            last_m  = machine_last_part.get(fixed_machine)
            purge   = (
                last_m is not None and last_m != part
                and part_color.get(last_m, "UNKNOWN") != color
                and part_color.get(last_m, "UNKNOWN") != "UNKNOWN"
                and color != "UNKNOWN"
            )

            machine_hours[fixed_machine]   = round(co_hrs + run_hrs, 4)
            current_inventory[part]        = round(current_inventory.get(part, 0) + qty, 0)
            machine_last_part[fixed_machine] = part
            already_planned.add(part)

            inv_b = inventory.get(part, 0)
            days_b = round(inv_b / daily, 2) if daily > 0 else 0

            plan.append({
                "Part":             part,
                "Color":            color,
                "Category":         cat,
                "Fixed_Machine":    fixed_machine,
                "Fixed_Used":       "YES — Phase A (dedicated)",
                "Machine":          fixed_machine,
                "Run_Hours":        round(run_hrs, 3),
                "Changeover_Hrs":   round(co_hrs, 3),
                "Total_Hrs_Used":   round(co_hrs + run_hrs, 3),
                "Rate_Per_Hour":    round(r_val, 2),
                "Production_Qty":   qty,
                "Monthly_Indent":   round(monthly, 0),
                "Daily_Indent":     round(daily, 2),
                "Today_Target":     round(today_target_qty.get(part, 0), 0),
                "Changeover":       "No" if co_hrs == 0 else "Yes",
                "Color_Purge":      "Yes" if purge else "No",
                "Type":             "Fixed-Phase-A [DEDICATED]",
                "Role":             "Primary",
                "Tools_Available":  tools_available.get(part, 1),
                "Tools_Used":       1,
                "Runner_Lock":      "No",
                "Priority_Score":   score,
                "Phase":            1,
                "Indent_Met":       "YES" if qty >= max(0.0, daily - inv_b) - 0.5 else "NO",
                "Stagger_Adjusted": "No",
            })
            print(f"    Phase A → {part:30s} run={run_hrs:.2f}h  "
                  f"qty={qty:.0f}  days_before={days_b:.2f}  [DEDICATED]")

        else:
            # Phase B: run fixed part for min hours, then open to scheduling
            # Plan all active fixed parts on this machine (cover their daily indent)
            # then fill remaining hours with normal scheduling

            # Sort fixed parts: lowest days-coverage first, then highest hrs-needed
            def fp_sort(p):
                daily_p = indent_daily.get(p, 0)
                inv_p   = current_inventory.get(p, 0)
                r_p     = rate.get(p, 1)
                dc      = inv_p / daily_p if daily_p > 0 else 999
                hn      = daily_p / r_p if r_p > 0 else 0
                return (round(dc, 4), -round(hn, 4))

            sorted_fixed = sorted(active_fixed, key=fp_sort)

            # Calculate total hours needed for all fixed parts
            total_fixed_hrs = sum(
                fixed_part_hours(p)[0] for p in sorted_fixed
            )

            # Available hours for normal parts after fixed parts served
            remaining_for_normal = max(0.0, AVAILABLE_HOURS - total_fixed_hrs)

            print(f"    Phase B → fixed hrs needed: {total_fixed_hrs:.2f}h  "
                  f"remaining for others: {remaining_for_normal:.2f}h")

            # Use plan_machine_parts to cover fixed parts first
            fixed_rows = plan_machine_parts(
                fixed_machine,
                sorted(sorted_fixed, key=lambda p: priority_scores.get(p, 0), reverse=True),
                machine_hours, machine_last_part, current_inventory,
                plan, already_planned, priority_scores, scenario_id,
                available_cap=AVAILABLE_HOURS,
            )

            for row in fixed_rows:
                row["Type"]       = "Fixed-Phase-B"
                row["Fixed_Used"] = "YES — Phase B"

            if fixed_rows:
                print(f"    Phase B fixed parts planned on {fixed_machine}:")
                for row in fixed_rows:
                    print(f"      {row['Part']:30s} run={row['Run_Hours']:.2f}h  "
                          f"qty={row['Production_Qty']:.0f}")

    # ── PASS 1: Normal priority scheduling ───────────────────
    sorted_active = sorted(
        [p for p in active_parts if p not in already_planned],
        key=lambda p: priority_scores.get(p, 0),
        reverse=True,
    )

    print(f"\n  PRIMARY SCHEDULING PASS  ({len(sorted_active)} parts)")
    print(f"  {'Part':<30} {'Color':<10} {'Fixed':<20} {'Score':>6} "
          f"{'Days':>5} {'Status':<15} {'Machine(s)':<25} {'Run':>5} {'Qty':>8}")
    print(f"  {'─'*130}")

    for part in sorted_active:
        inv_now  = current_inventory.get(part, 0)
        daily    = indent_daily.get(part, 0)
        monthly  = indent_monthly.get(part, 0)
        score    = priority_scores.get(part, 0)
        tools    = tools_available.get(part, 1)
        category = part_category.get(part, "Stranger")
        color    = part_color.get(part, "UNKNOWN")
        fixed_m  = part_fixed_machine.get(part, "—")
        days_cov = inv_now / daily if daily > 0 else 999
        buf_lbl  = (
            "CRITICAL"     if inv_now == 0 else
            "BELOW_SAFETY" if days_cov < SAFETY_DAYS else
            "BUILDING"     if days_cov < TARGET_DAYS else
            "AT_TARGET"
        )

        if monthly == 0:
            deferred.append({"Part": part, "Color": color, "Fixed_Machine": fixed_m,
                             "Category": category, "Reason": "No indent"})
            print(f"  {part:<30} {color:<10} {fixed_m:<20} {score:>6.1f} {days_cov:>5.1f} "
                  f"{'DEFERRED':<15}  —")
            continue

        t_blk, t_rsn = terminal_blocked(part)
        if t_blk:
            not_planned.append({"Part": part, "Color": color, "Fixed_Machine": fixed_m,
                                "Category": category, "Score": score, "Reason": t_rsn,
                                "Action_Needed": "Replenish terminal"})
            print(f"  {part:<30} {color:<10} {fixed_m:<20} {score:>6.1f} {days_cov:>5.1f} "
                  f"{buf_lbl:<15}  ✗ TERMINAL ZERO STOCK")
            continue

        if not compat.get(part):
            not_planned.append({"Part": part, "Color": color, "Fixed_Machine": fixed_m,
                                "Category": category, "Score": score,
                                "Reason": "Not in matrix", "Action_Needed": "Add to matrix"})
            print(f"  {part:<30} {color:<10} {fixed_m:<20} {score:>6.1f} {days_cov:>5.1f} "
                  f"{buf_lbl:<15}  ✗ NOT IN MATRIX")
            continue

        new_rows = assign_part(
            part, scenario_id, machine_hours, machine_last_part,
            current_inventory, plan, already_planned, priority_scores,
        )

        if new_rows:
            plan.extend([r for r in new_rows if r not in plan])
            total_qty  = sum(float(r["Production_Qty"]) for r in new_rows)
            total_run  = sum(float(r["Run_Hours"]) for r in new_rows)
            mach_str   = ", ".join(r["Machine"] for r in new_rows)
            flag       = " [MULTI]"    if len(new_rows) > 1 else ""
            flag      += " [ZERO-INV]" if inv_now == 0 else ""
            purges     = sum(1 for r in new_rows if r.get("Color_Purge") == "Yes")
            flag      += f" [PURGE×{purges}]" if purges > 0 else ""
            fu_tags    = {r.get("Fixed_Used","") for r in new_rows}
            if "YES" in fu_tags:       flag += " [FIXED ✓]"
            elif "FALLBACK" in fu_tags: flag += " [FIXED FALLBACK]"
            print(f"  {part:<30} {color:<10} {fixed_m:<20} {score:>6.1f} {days_cov:>5.1f} "
                  f"{buf_lbl:<15}  {mach_str:<25}  {total_run:>5.2f}  {total_qty:>8.0f}  ✓{flag}")
        else:
            not_planned.append({
                "Part": part, "Color": color, "Fixed_Machine": fixed_m,
                "Category": category, "Score": score, "Days_Coverage": round(days_cov, 2),
                "Buffer_Status": buf_lbl, "Daily_Indent": round(daily, 2),
                "Inventory_Now": round(inv_now, 0),
                "Compatible_Machines": ", ".join(compat.get(part, [])),
                "Reason": "No compatible machine has capacity",
                "Action_Needed": "Review matrix or add machines",
            })
            print(f"  {part:<30} {color:<10} {fixed_m:<20} {score:>6.1f} {days_cov:>5.1f} "
                  f"{buf_lbl:<15}  ✗ NO CAPACITY")

    # ── PASS 2: Displacement pass ─────────────────────────────
    # Priority:
    # 1. Runners with inv < RUNNER_DISPLACE_DAYS × daily (sub-N-day)
    # 2. Repeaters with inv = 0
    # Both can displace Strangers and Repeaters with >= REPEATER_VICTIM_DAYS stock

    print(f"\n  DISPLACEMENT PASS")

    # Collect displacement candidates
    runner_cands = sorted(
        [
            p for p in active_parts
            if p not in already_planned
            and part_category.get(p, "Stranger") == "Runner"
            and current_inventory.get(p, 0) < RUNNER_DISPLACE_DAYS * indent_daily.get(p, 0)
            and compat.get(p)
        ],
        key=lambda p: indent_daily.get(p, 0), reverse=True,
    )
    repeater_zero_cands = sorted(
        [
            p for p in active_parts
            if p not in already_planned
            and part_category.get(p, "Stranger") == "Repeater"
            and current_inventory.get(p, 0) == 0
            and compat.get(p)
        ],
        key=lambda p: indent_daily.get(p, 0), reverse=True,
    )

    all_disp_cands = runner_cands + repeater_zero_cands

    if all_disp_cands:
        print(f"  Candidates: {len(runner_cands)} runners (sub-{RUNNER_DISPLACE_DAYS}-day)  "
              f"+ {len(repeater_zero_cands)} zero-inv repeaters")
        for part in all_disp_cands:
            inv   = current_inventory.get(part, 0)
            daily = indent_daily.get(part, 0)
            cat   = part_category.get(part, "Stranger")
            print(f"    {part:<30} [{cat}]  inv={inv:.0f}  daily={daily:.2f}")

        for part in all_disp_cands:
            success = displace_and_assign(
                part, machine_hours, machine_last_part,
                current_inventory, plan, already_planned,
                priority_scores, scenario_id,
            )
            if success:
                inv_now  = current_inventory.get(part, 0)
                daily    = indent_daily.get(part, 0)
                cat      = part_category.get(part, "Stranger")
                disp_log.append({
                    "Part":     part,
                    "Category": cat,
                    "Result":   "PLACED",
                    "Inv_After":round(inv_now, 0),
                })
                not_planned[:] = [r for r in not_planned if r.get("Part") != part]
            else:
                print(f"    ✗ {part}  — displacement failed (no viable machine)")
                disp_log.append({
                    "Part":     part,
                    "Category": part_category.get(part, "Stranger"),
                    "Result":   "FAILED — no viable machine",
                    "Inv_After": round(current_inventory.get(part, 0), 0),
                })
    else:
        print(f"  No displacement candidates today  ✓")

    # ── Utilisation enforcer ─────────────────────────────────
    micro_idle = utilization_enforcer(
        plan, machine_hours, machine_last_part,
        list(parts), already_planned,
        current_inventory, scenario_id, priority_scores,
        fixed_machines_today=fixed_machines_phase_a,
    )

    # ── CO stagger ───────────────────────────────────────────
    print(f"\n  CO Stagger  (min gap = {MIN_CO_GAP_HRS*60:.0f} min)")
    n_adj = stagger_co_by_quantity(plan, all_machines, scenario_id, current_inventory)
    print(f"    {'No adjustments  ✓' if n_adj == 0 else str(n_adj)+' adjustment(s)'}")
    stagger_changeovers_serial_queue(plan, all_machines)

    # ── Build output dataframes ──────────────────────────────
    multi_df       = build_multi_machine_view(plan)
    pvsi_df        = build_production_vs_indent(plan, list(parts))
    inv_tgt_df     = build_inventory_target_sheet(plan, list(parts), scenario_id)
    fixed_adh_df   = build_fixed_adherence_sheet(plan)
    disp_df        = pd.DataFrame(disp_log) if disp_log else pd.DataFrame()

    # Indent status
    indent_rows = []
    for row in plan:
        p       = row["Part"]
        planned = float(row.get("Production_Qty") or 0)
        daily   = indent_daily.get(p, 0)
        inv_b   = inventory.get(p, 0)
        meets   = planned >= daily
        indent_rows.append({
            "Part":               p,
            "Color":              part_color.get(p, "UNKNOWN"),
            "Category":           part_category.get(p, "Stranger"),
            "Fixed_Machine":      part_fixed_machine.get(p, "—"),
            "Machine":            row.get("Machine", "—"),
            "Color_Purge":        row.get("Color_Purge", "No"),
            "Role":               row.get("Role", "Primary"),
            "Priority_Score":     round(row.get("Priority_Score", 0), 2),
            "Buffer_Status":      (
                "CRITICAL"     if inv_b == 0 else
                "BELOW_SAFETY" if (daily > 0 and inv_b / daily < SAFETY_DAYS)
                else "BUILDING" if (daily > 0 and inv_b / daily < TARGET_DAYS)
                else "AT_TARGET"
            ),
            "Run_Hours":          round(float(row.get("Run_Hours") or 0), 2),
            "Planned_Qty":        round(planned, 0),
            "Daily_Indent":       round(daily, 2),
            "Gap_vs_Daily":       round(daily - planned, 0),
            "Meets_Daily_Indent": "YES ✓" if meets else "NO ✗",
            "Inventory_Before":   round(inv_b, 0),
            "Total_Available":    round(inv_b + planned, 0),
            "Covers_With_Inv":    "YES ✓" if (inv_b + planned) >= daily else "NO ✗",
        })
    indent_status_df = pd.DataFrame(indent_rows)
    if not indent_status_df.empty:
        indent_status_df = indent_status_df.sort_values(
            ["Meets_Daily_Indent", "Gap_vs_Daily"], ascending=[True, False]
        ).reset_index(drop=True)

    # Inventory health
    inv_rows = []
    for p in parts:
        inv_b    = inventory.get(p, 0)
        daily    = indent_daily.get(p, 0)
        produced = sum(float(r["Production_Qty"]) for r in plan if r["Part"] == p)
        inv_af   = inv_b + produced
        dc       = inv_af / daily if daily > 0 else 0
        inv_rows.append({
            "Part":            p,
            "Color":           part_color.get(p, "UNKNOWN"),
            "Fixed_Machine":   part_fixed_machine.get(p, "—"),
            "Category":        part_category.get(p, "Stranger"),
            "Tools":           tools_available.get(p, 1),
            "Rate_Per_Hour":   round(rate.get(p, 0), 2),
            "Daily_Indent":    round(daily, 2),
            "Inv_Before":      round(inv_b, 0),
            "Produced_Today":  round(produced, 0),
            "Inv_After":       round(inv_af, 0),
            "Days_Coverage":   round(dc, 2),
            "Safety_Floor":    SAFETY_DAYS,
            "Target_Ceiling":  TARGET_DAYS,
            "Status":          (
                "AT_TARGET" if dc >= TARGET_DAYS else
                "OK"        if dc >= SAFETY_DAYS else
                "LOW"       if dc >= 1 else "CRITICAL"
            ),
        })

    # Machine utilisation
    mach_rows = []
    for m in all_machines:
        used      = machine_hours.get(m, 0)
        parts_run = list({r["Part"] for r in plan if r["Machine"] == m})
        co_count  = sum(1 for r in plan if r["Machine"] == m and r.get("Changeover") == "Yes")
        purges    = sum(1 for r in plan if r["Machine"] == m and r.get("Color_Purge") == "Yes")
        colors_m  = list({part_color.get(r["Part"],"UNKNOWN") for r in plan if r["Machine"] == m})
        fixed_on_m = [p for p in parts_run if part_fixed_machine.get(p) == m]
        util_pct  = round(used / AVAILABLE_HOURS * 100, 1)
        phase_today = "A" if m in fixed_machines_phase_a else ("B" if m in machine_fixed_parts else "—")
        mach_rows.append({
            "Machine":              m,
            "Fixed_Parts":          ", ".join(fixed_on_m) if fixed_on_m else "—",
            "Fixed_Phase_Today":    phase_today,
            "Colors_Today":         ", ".join(sorted(colors_m)),
            "Color_Purges":         purges,
            "Used_Hours":           round(used, 2),
            "Unused_Hours":         round(AVAILABLE_HOURS - used, 2),
            "Utilization_%":        util_pct,
            "Status":               (
                "FULL"      if used >= AVAILABLE_HOURS - 0.3 else
                "GOOD"      if used >= AVAILABLE_HOURS * 0.98 else
                "OK"        if used >= AVAILABLE_HOURS * 0.90 else
                "PARTIAL"   if used >= AVAILABLE_HOURS * 0.85 else
                "UNDERUSED"
            ),
            "Parts_Planned":        len(parts_run),
            "Changeovers":          co_count,
            "Last_Part_Run":        machine_last_part.get(m) or "—",
            "All_Parts":            ", ".join(parts_run) if parts_run else "—",
        })

    micro_df   = pd.DataFrame(micro_idle) if micro_idle else pd.DataFrame()
    plan_df    = pd.DataFrame(plan)       if plan       else pd.DataFrame()
    def_df     = pd.DataFrame(deferred)   if deferred   else pd.DataFrame()
    not_df     = pd.DataFrame(not_planned) if not_planned else pd.DataFrame()
    mach_df    = pd.DataFrame(mach_rows)
    inv_df     = pd.DataFrame(inv_rows)

    if not plan_df.empty:
        for i, (col, val) in enumerate([
            ("Planning_Date", str(PLANNING_DATE)),
            ("Scenario",      scenario_desc),
            ("Working_Days",  WORKING_DAYS),
            ("Safety_Days",   SAFETY_DAYS),
            ("Target_Days",   TARGET_DAYS),
            ("OPD_Cap",       cap_today),
        ]):
            plan_df.insert(i, col, val)

    # Summary print
    fa_used  = (fixed_adh_df["Status"] == "FIXED MACHINE USED").sum() if not fixed_adh_df.empty else 0
    fa_fallb = fixed_adh_df["Status"].str.contains("FALLBACK", na=False).sum() if not fixed_adh_df.empty else 0
    fa_notpl = (fixed_adh_df["Status"] == "NOT PLANNED TODAY").sum() if not fixed_adh_df.empty else 0
    disp_placed = (disp_df["Result"] == "PLACED").sum() if not disp_df.empty else 0
    disp_failed = disp_df["Result"].str.contains("FAILED", na=False).sum() if not disp_df.empty else 0

    print(f"\n  {'='*65}")
    print(f"  SUMMARY — {scenario_desc}")
    print(f"    Parts planned           : {len(already_planned)}")
    print(f"    Fixed machine used      : {fa_used}  |  fallback: {fa_fallb}  |  not planned: {fa_notpl}")
    print(f"    Displacement placed     : {disp_placed}  |  failed: {disp_failed}")
    print(f"    Not planned             : {len(not_planned)}")
    print(f"    Deferred                : {len(deferred)}")
    if not mach_df.empty:
        print(f"    Avg utilization         : {mach_df['Utilization_%'].mean():.1f}%")
        print(f"    Colour purges           : {mach_df['Color_Purges'].sum()}")
    if not inv_tgt_df.empty:
        for s in ["AT_TARGET", "BUILDING", "BELOW_SAFETY", "CRITICAL"]:
            n = (inv_tgt_df["Buffer_Status"] == s).sum()
            print(f"    {s:<18} : {n}")
    print(f"  {'='*65}")

    return (
        plan_df, def_df, not_df, mach_df, inv_df,
        machine_last_part, horizon_df, indent_status_df,
        score_df, micro_df, multi_df, pvsi_df,
        inv_tgt_df, fixed_adh_df, disp_df,
    )

# =============================================================
# SECTION 22 — PART AUDIT
# =============================================================

all_parts_raw = list(data["Material"].unique())
matrix_parts  = set(str(p).strip() for p in matrix_df["Part"] if pd.notna(p))
zero_rate_set = set(data_zero_rate["Material"].unique())

audit_rows = []
for part in all_parts_raw:
    inv     = inventory.get(part, 0.0)
    r_val   = rate.get(part, None)
    monthly = indent_monthly.get(part, 0.0)
    daily   = indent_daily.get(part, 0.0)
    row_d   = data[data["Material"] == part]
    ct_raw  = row_d[c_ct].values[0]   if len(row_d) else "—"
    cv_raw  = row_d[c_cav].values[0]  if len(row_d) else "—"
    tools   = tools_available.get(part, 1)
    color   = part_color.get(part, "UNKNOWN")
    fixed_m = part_fixed_machine.get(part, "—")
    days_cov = inv / daily if daily > 0 else 0

    part_terms    = part_terminals.get(part, [])
    t_blk, t_rsn  = terminal_blocked(part)
    terms_str     = ", ".join(part_terms) if part_terms else "—"
    terms_zero    = (
        ", ".join(sorted([
            f"{t}(inv={terminal_status.get(t,0):.0f})"
            for t in part_terms if terminal_status.get(t, 0) <= 0
        ])) or "—"
    )

    if part in zero_rate_set or r_val is None:
        status, gate, reason = "ZERO/MISSING CYCLE TIME", "GATE 1", f"CT={ct_raw}"
    elif part not in matrix_parts:
        status, gate, reason = "NOT IN MATRIX", "GATE 2", "No compatible machine"
    elif monthly == 0:
        status, gate, reason = "ZERO/MISSING INDENT", "GATE 3", "Indent = 0"
    elif daily <= MIN_DAILY_INDENT:
        status, gate, reason = "SKIPPED (LOW INDENT)", "GATE 4a", f"Daily ≤ {MIN_DAILY_INDENT}"
    elif (monthly / r_val if r_val else 0) <= MIN_INDENT_HOURS:
        status, gate, reason = "SKIPPED (TRIVIAL RUN)", "GATE 4b", f"Monthly hrs ≤ {MIN_INDENT_HOURS}h"
    elif daily > 0 and inv >= TARGET_DAYS * daily:
        status, gate, reason = f"AT TARGET — SKIP", "GATE 5", f"Inv ≥ {TARGET_DAYS}×daily"
    elif t_blk:
        status, gate, reason = "BLOCKED — TERMINAL ZERO", "GATE 6", t_rsn
    else:
        status, gate, reason = "ENTERS SCHEDULER", "—", "Passed all gates"

    audit_rows.append({
        "Part":               part,
        "Color":              color,
        "Fixed_Machine":      fixed_m,
        "Category":           part_category.get(part, "Stranger"),
        "Gate_Failed":        gate,
        "Reason":             reason,
        "Terminals_Required": terms_str,
        "Terminals_Zero_Inv": terms_zero,
        "Monthly_Indent":     round(monthly, 0),
        "Daily_Indent":       round(daily, 2),
        "Inventory":          round(inv, 0),
        "Days_Coverage":      round(days_cov, 2),
        "Tools":              tools,
        "Rate_Per_Hour":      round(r_val, 2) if r_val else "—",
        "Cycle_Time":         ct_raw,
        "Cavity":             cv_raw,
        "Status":             status,
    })

audit_df    = pd.DataFrame(audit_rows)
gate_counts = audit_df["Status"].value_counts()
print(f"\n  Part audit ({len(all_parts_raw)} total):")
for s, cnt in gate_counts.items():
    print(f"    {'✓' if s=='ENTERS SCHEDULER' else '·'}  {s:<50}: {cnt:>4}")

vt_terminal_status_df = build_terminal_status_sheet()
if not vt_terminal_status_df.empty:
    z = (vt_terminal_status_df["Available_Today"] == "NO — ZERO STOCK").sum()
    b = int(vt_terminal_status_df["Parts_Blocked_Count"].sum())
    print(f"\n  Terminals: {len(vt_terminal_status_df)} tracked  |  {z} zero  |  {b} parts blocked")

# Identify active parts (from data_valid that are in matrix)
active_parts_global = data_valid[
    data_valid["Material"].isin(matrix_df["Part"])
]["Material"].unique()

# =============================================================
# SECTION 23 — RUN
# =============================================================

(plan_df, def_df, not_df, mach_df, inv_df,
 final_state, horizon_df, indent_status_df,
 score_df, micro_df, multi_df, pvsi_df,
 inv_tgt_df, fixed_adh_df, disp_df) = schedule(
    active_parts_global, f"{SECTION_LABEL} Machines"
)

save_machine_state(final_state)

# =============================================================
# SECTION 24 — MACHINE-WISE PLAN  +  CO QUEUE
# =============================================================

def build_machine_wise_plan(plan_df_in):
    if plan_df_in.empty:
        return pd.DataFrame()

    def sf(val, default=0.0):
        try:
            v = float(val)
            return v if not np.isnan(v) else default
        except (TypeError, ValueError):
            return default

    rows = []
    for m in all_machines:
        mr = plan_df_in[plan_df_in["Machine"] == m].copy()
        if mr.empty:
            continue
        for _, pr in mr.iterrows():
            p = pr.get("Part", "—")
            rows.append({
                "Machine":           m,
                "Part":              p,
                "Color":             part_color.get(p, "UNKNOWN"),
                "Category":          part_category.get(p, "Stranger"),
                "Fixed_Machine":     part_fixed_machine.get(p, "—"),
                "Fixed_Used":        pr.get("Fixed_Used", "N/A"),
                "Role":              pr.get("Role", "Primary"),
                "Tools_Available":   pr.get("Tools_Available", 1),
                "Priority_Score":    round(sf(pr.get("Priority_Score", 0)), 2),
                "Rate_Per_Hour":     round(sf(pr.get("Rate_Per_Hour", 0)), 2),
                "Run_Hours":         round(sf(pr.get("Run_Hours", 0)), 2),
                "Changeover_Hrs":    round(sf(pr.get("Changeover_Hrs", 0)), 3),
                "Changeover_Needed": pr.get("Changeover", "No") or "No",
                "Color_Purge":       pr.get("Color_Purge", "No") or "No",
                "Production_Qty":    round(sf(pr.get("Production_Qty", 0)), 0),
                "Daily_Indent":      round(sf(pr.get("Daily_Indent", 0)), 2),
                "Today_Target":      round(sf(pr.get("Today_Target", 0)), 0),
                "Monthly_Indent":    round(sf(pr.get("Monthly_Indent", 0)), 0),
                "Type":              pr.get("Type", "Primary") or "Primary",
                "Row_Type":          "Part",
            })

        co_t    = mr["Changeover_Hrs"].apply(lambda x: sf(x, 0)).sum()
        run_t   = mr["Run_Hours"].apply(lambda x: sf(x, 0)).sum()
        qty_t   = mr["Production_Qty"].apply(lambda x: sf(x, 0)).sum()
        co_cnt  = int(mr["Changeover"].eq("Yes").sum())
        pu_cnt  = int(mr["Color_Purge"].eq("Yes").sum()) if "Color_Purge" in mr.columns else 0
        hrs_t   = round(co_t + run_t, 2)
        cols_l  = sorted({part_color.get(p, "UNKNOWN") for p in mr["Part"]})
        fix_l   = [p for p in mr["Part"] if part_fixed_machine.get(p) == m]

        rows.append({
            "Machine":           m,
            "Part":              f"TOTAL — {m}",
            "Color":             ", ".join(cols_l),
            "Category":          "—",
            "Fixed_Machine":     ", ".join(fix_l) if fix_l else "—",
            "Fixed_Used":        "—",
            "Role":              "—",
            "Tools_Available":   "—",
            "Priority_Score":    "—",
            "Rate_Per_Hour":     "—",
            "Run_Hours":         round(run_t, 2),
            "Changeover_Hrs":    round(co_t, 2),
            "Changeover_Needed": f"{co_cnt} changeover(s)",
            "Color_Purge":       f"{pu_cnt} purge(s)",
            "Production_Qty":    round(qty_t, 0),
            "Daily_Indent":      "—",
            "Today_Target":      "—",
            "Monthly_Indent":    "—",
            "Type":              (
                f"Total {hrs_t}h / {AVAILABLE_HOURS}h  |  "
                f"Idle {round(AVAILABLE_HOURS - hrs_t, 2)}h  |  "
                f"Util {round(hrs_t / AVAILABLE_HOURS * 100, 1)}%"
            ),
            "Row_Type":          "Summary",
        })
        rows.append({k: "" for k in rows[-1].keys()})
    return pd.DataFrame(rows)


def build_co_queue_df(plan, machines):
    events = _collect_co_events(plan, machines)
    if not events:
        return pd.DataFrame()
    events.sort(key=lambda e: e["natural_start"])
    rows = []
    tc_free = 0.0
    for pos, ev in enumerate(events, 1):
        ns   = _recompute_natural_start(ev, plan)
        co_h = ev["co_duration"]
        as_  = max(ns, tc_free)
        wait = round((as_ - ns) * 60, 1)
        tc_free = as_ + co_h
        bc   = part_color.get(ev["part_before"], "UNKNOWN")
        ac   = part_color.get(ev["part_after"],  "UNKNOWN")
        cc   = bc != ac and bc != "UNKNOWN" and ac != "UNKNOWN"
        note = ""
        if wait > 0:
            note += f"Wait ~{wait}min — keep running previous part."
        if cc:
            note += (("  " if note else "") +
                     f"COLOUR CHANGE {bc}→{ac} — 10min purge.")
        rows.append({
            "Queue_Position":  pos,
            "Machine":         ev["machine"],
            "Part_Before":     ev["part_before"],
            "Color_Before":    bc,
            "Part_After":      ev["part_after"],
            "Color_After":     ac,
            "Color_Change":    "YES — PURGE" if cc else "No",
            "CO_Duration_Min": round(co_h * 60, 1),
            "Note":            note if note else "TC available immediately. No colour change.",
        })
    return pd.DataFrame(rows)


mw_df    = build_machine_wise_plan(plan_df)
co_q_df  = build_co_queue_df(
    [{k: v for k, v in r.items()} for r in plan_df.to_dict("records")]
    if not plan_df.empty else [],
    all_machines,
)

# =============================================================
# SECTION 25 — EXCEL OUTPUT
# =============================================================

from openpyxl import load_workbook
from openpyxl.styles import PatternFill, Font, Alignment
from openpyxl.utils import get_column_letter

S = SECTION_LABEL   # shorthand for sheet name prefix

HEADER_COLORS = {
    f"{S}_Plan_By_Machine":      "0D6E6E",
    f"{S}_CO_Queue":             "375623",
    f"{S}_Plan":                 "1F4E79",
    f"{S}_Daily_Indent_Status":  "0F4C2A",
    f"{S}_Multi_Machine_Parts":  "4A235A",
    f"{S}_Production_vs_Indent": "154360",
    f"{S}_Inventory_Target":     "1B4F72",
    f"{S}_Priority_Scores":      "2C4770",
    f"{S}_Machine_Util":         "375623",
    f"{S}_Not_Planned":          "7B2C2C",
    f"{S}_Deferred":             "7F6000",
    f"{S}_Inventory_Health":     "4A235A",
    f"{S}_Indent_Horizon":       "154360",
    f"{S}_Part_Audit":           "1C3557",
    f"{S}_Micro_Idle":           "5C3D2E",
    f"{S}_Terminal_Status":      "7B1C1C",
    f"{S}_Fixed_Adherence":      "1A5276",
    f"{S}_Displacement_Log":     "1B4F0A",
}

STATUS_FILLS = {
    "FULL":         PatternFill("solid", fgColor="C6EFCE"),
    "GOOD":         PatternFill("solid", fgColor="DDEBF7"),
    "OK":           PatternFill("solid", fgColor="EBF5E1"),
    "PARTIAL":      PatternFill("solid", fgColor="FFEB9C"),
    "UNDERUSED":    PatternFill("solid", fgColor="FFC7CE"),
    "AT_TARGET":    PatternFill("solid", fgColor="C6EFCE"),
    "BUILDING":     PatternFill("solid", fgColor="DDEBF7"),
    "BELOW_SAFETY": PatternFill("solid", fgColor="FFEB9C"),
    "CRITICAL":     PatternFill("solid", fgColor="FFC7CE"),
    "LOW":          PatternFill("solid", fgColor="FFEB9C"),
    "YES ✓":        PatternFill("solid", fgColor="C6EFCE"),
    "NO ✗":         PatternFill("solid", fgColor="FFC7CE"),
    "OVER":         PatternFill("solid", fgColor="DDEBF7"),
    "UNDER":        PatternFill("solid", fgColor="FFC7CE"),
    "MET":          PatternFill("solid", fgColor="C6EFCE"),
    "YES — PURGE":  PatternFill("solid", fgColor="FFC7CE"),
    "Yes":          PatternFill("solid", fgColor="FFEB9C"),
    "ZERO/MISSING CYCLE TIME":              PatternFill("solid", fgColor="FFC7CE"),
    "NOT IN MATRIX":                        PatternFill("solid", fgColor="FFEB9C"),
    "ZERO/MISSING INDENT":                  PatternFill("solid", fgColor="FFEB9C"),
    "ENTERS SCHEDULER":                     PatternFill("solid", fgColor="C6EFCE"),
    "YES":                                  PatternFill("solid", fgColor="C6EFCE"),
    "NO — ZERO STOCK":                      PatternFill("solid", fgColor="FFC7CE"),
    "BLOCKING — reschedule":                PatternFill("solid", fgColor="FFC7CE"),
    "No impact today":                      PatternFill("solid", fgColor="C6EFCE"),
    "BLOCKED — TERMINAL ZERO":              PatternFill("solid", fgColor="FFC7CE"),
    "FIXED MACHINE USED":                   PatternFill("solid", fgColor="C6EFCE"),
    "FIXED + FALLBACK USED":                PatternFill("solid", fgColor="DDEBF7"),
    "FALLBACK ONLY — fixed had no capacity":PatternFill("solid", fgColor="FFEB9C"),
    "NOT PLANNED TODAY":                    PatternFill("solid", fgColor="EDEDED"),
    "PLACED":                               PatternFill("solid", fgColor="C6EFCE"),
    "FAILED — no viable machine":           PatternFill("solid", fgColor="FFC7CE"),
    "AT TARGET — SKIP":                     PatternFill("solid", fgColor="C6EFCE"),
}

STATUS_TRIGGER_COLS = [
    "Status", "Indent_Status", "Meets_Daily", "Covers_With",
    "Gap_Direction", "Buffer_Status", "Scheduled_Today",
    "Color_Change", "Color_Purge", "Available_Today", "Impact",
    "Result", "Fixed_Used",
]


def style_sheet(ws, header_hex):
    for cell in ws[1]:
        cell.fill      = PatternFill("solid", fgColor=header_hex)
        cell.font      = Font(bold=True, color="FFFFFF", size=11)
        cell.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)
    ws.row_dimensions[1].height = 32
    for col in ws.columns:
        cl     = get_column_letter(col[0].column)
        max_l  = max((len(str(c.value)) for c in col if c.value is not None), default=8)
        ws.column_dimensions[cl].width = max(10, min(55, max_l + 3))
    headers = [cell.value for cell in ws[1]]
    for ci, cn in enumerate(headers, 1):
        if cn and any(x in str(cn) for x in STATUS_TRIGGER_COLS):
            for row in ws.iter_rows(min_row=2, min_col=ci, max_col=ci):
                for cell in row:
                    fill = STATUS_FILLS.get(str(cell.value))
                    if fill:
                        cell.fill = fill
    ws.freeze_panes = "A2"


def style_machine_wise(ws):
    hdr_fill  = PatternFill("solid", fgColor="1F4E79")
    sum_fill  = PatternFill("solid", fgColor="0D9488")
    pt_fills  = [PatternFill("solid", fgColor="EFF6FF"), PatternFill("solid", fgColor="F0FDF4")]
    co_fill   = PatternFill("solid", fgColor="FEF9C3")
    pu_fill   = PatternFill("solid", fgColor="FFC7CE")
    fix_fill  = PatternFill("solid", fgColor="E8F4FD")

    for cell in ws[1]:
        cell.fill = hdr_fill
        cell.font = Font(bold=True, color="FFFFFF", size=11)
        cell.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)
    ws.row_dimensions[1].height = 30

    headers  = [c.value for c in ws[1]]
    rt_col   = headers.index("Row_Type")          + 1 if "Row_Type"          in headers else None
    co_col   = headers.index("Changeover_Needed") + 1 if "Changeover_Needed" in headers else None
    pu_col   = headers.index("Color_Purge")       + 1 if "Color_Purge"       in headers else None
    mc_col   = headers.index("Machine")           + 1 if "Machine"           in headers else None
    fu_col   = headers.index("Fixed_Used")        + 1 if "Fixed_Used"        in headers else None

    mc_idx   = 0
    cur_mc   = None
    for row in ws.iter_rows(min_row=2):
        rt = row[rt_col-1].value if rt_col else ""
        mc = row[mc_col-1].value if mc_col else ""
        if mc and mc != cur_mc:
            cur_mc = mc
            mc_idx = (mc_idx + 1) % 2
        if rt == "Summary":
            for cell in row:
                cell.fill = sum_fill
                cell.font = Font(bold=True, color="FFFFFF", size=11)
                cell.alignment = Alignment(horizontal="center", vertical="center")
        elif rt == "Part":
            base = pt_fills[mc_idx]
            for cell in row:
                cell.fill = base
                cell.alignment = Alignment(vertical="center")
            if fu_col and str(row[fu_col-1].value).startswith("YES"):
                for cell in row:
                    cell.fill = fix_fill
            if co_col and str(row[co_col-1].value) == "Yes":
                row[co_col-1].fill = co_fill
            if pu_col and str(row[pu_col-1].value) == "Yes":
                row[pu_col-1].fill = pu_fill

    for col in ws.columns:
        cl = get_column_letter(col[0].column)
        ml = max((len(str(c.value)) for c in col if c.value), default=8)
        ws.column_dimensions[cl].width = max(10, min(45, ml + 3))
    ws.freeze_panes = "B2"


def style_inv_target(ws):
    style_sheet(ws, HEADER_COLORS[f"{S}_Inventory_Target"])
    headers  = [c.value for c in ws[1]]
    sc       = headers.index("Buffer_Status") + 1 if "Buffer_Status" in headers else None
    fills    = {
        "CRITICAL":     PatternFill("solid", fgColor="FFD7D7"),
        "BELOW_SAFETY": PatternFill("solid", fgColor="FFF2CC"),
        "BUILDING":     PatternFill("solid", fgColor="DDEEFF"),
        "AT_TARGET":    PatternFill("solid", fgColor="E2EFDA"),
    }
    for row in ws.iter_rows(min_row=2):
        if not sc:
            continue
        s   = str(row[sc-1].value)
        f   = fills.get(s)
        if f:
            for cell in row:
                if cell.fill.fill_type == "none" or cell.fill.fgColor.rgb in ("00000000","FFFFFFFF"):
                    cell.fill = f
        if s == "CRITICAL":
            for cell in row:
                cell.font = Font(bold=True)


def style_pvsi(ws):
    style_sheet(ws, HEADER_COLORS[f"{S}_Production_vs_Indent"])
    headers = [c.value for c in ws[1]]
    gc      = headers.index("Gap_Direction") + 1 if "Gap_Direction" in headers else None
    of      = PatternFill("solid", fgColor="DDEBF7")
    uf      = PatternFill("solid", fgColor="FFC7CE")
    mf      = PatternFill("solid", fgColor="C6EFCE")
    for row in ws.iter_rows(min_row=2):
        if gc:
            v = str(row[gc-1].value)
            if v == "OVER":   row[gc-1].fill = of
            elif v == "UNDER":
                row[gc-1].fill = uf
                for c in row: c.font = Font(bold=True)
            elif v == "MET":  row[gc-1].fill = mf


def style_co_queue(ws):
    style_sheet(ws, HEADER_COLORS[f"{S}_CO_Queue"])
    headers = [c.value for c in ws[1]]
    cc      = headers.index("Color_Change") + 1 if "Color_Change" in headers else None
    pf      = PatternFill("solid", fgColor="FFC7CE")
    for row in ws.iter_rows(min_row=2):
        if cc and str(row[cc-1].value).startswith("YES"):
            for cell in row:
                if cell.fill.fill_type == "none" or cell.fill.fgColor.rgb in ("00000000","FFFFFFFF"):
                    cell.fill = pf
            row[cc-1].font = Font(bold=True, color="7B1C1C")


def style_fixed_adherence(ws):
    style_sheet(ws, HEADER_COLORS[f"{S}_Fixed_Adherence"])
    headers = [c.value for c in ws[1]]
    sc      = headers.index("Status") + 1 if "Status" in headers else None
    if not sc:
        return
    for row in ws.iter_rows(min_row=2):
        s = str(row[sc-1].value)
        f = STATUS_FILLS.get(s)
        if f:
            for cell in row:
                if cell.fill.fill_type == "none" or cell.fill.fgColor.rgb in ("00000000","FFFFFFFF"):
                    cell.fill = f
        if s == "FALLBACK ONLY — fixed had no capacity":
            for cell in row: cell.font = Font(bold=True)


def style_multi_machine(ws):
    style_sheet(ws, HEADER_COLORS[f"{S}_Multi_Machine_Parts"])
    headers = [c.value for c in ws[1]]
    rc      = headers.index("Role") + 1 if "Role" in headers else None
    tf = PatternFill("solid", fgColor="0D9488")
    pf = PatternFill("solid", fgColor="EFF6FF")
    ef = PatternFill("solid", fgColor="FEF9C3")
    for row in ws.iter_rows(min_row=2):
        if not rc: continue
        r = str(row[rc-1].value)
        if r == "TOTAL":
            for cell in row:
                cell.fill = tf
                cell.font = Font(bold=True, color="FFFFFF", size=11)
        elif "Tool-Expansion" in r:
            for cell in row: cell.fill = ef
        elif r == "Primary":
            for cell in row: cell.fill = pf


# Write Excel
print(f"\nWriting output → {output_path}")

sheets = {
    f"{S}_Plan_By_Machine":      mw_df,
    f"{S}_CO_Queue":             co_q_df,
    f"{S}_Plan":                 plan_df,
    f"{S}_Fixed_Adherence":      fixed_adh_df,
    f"{S}_Displacement_Log":     disp_df,
    f"{S}_Inventory_Target":     inv_tgt_df,
    f"{S}_Multi_Machine_Parts":  multi_df,
    f"{S}_Production_vs_Indent": pvsi_df,
    f"{S}_Daily_Indent_Status":  indent_status_df,
    f"{S}_Priority_Scores":      score_df,
    f"{S}_Machine_Util":         mach_df,
    f"{S}_Not_Planned":          not_df,
    f"{S}_Deferred":             def_df,
    f"{S}_Inventory_Health":     inv_df,
    f"{S}_Indent_Horizon":       horizon_df,
    f"{S}_Part_Audit":           audit_df,
    f"{S}_Terminal_Status":      vt_terminal_status_df,
}
if not micro_df.empty:
    sheets[f"{S}_Micro_Idle"] = micro_df

with pd.ExcelWriter(output_path, engine="openpyxl") as writer:
    for sn, df in sheets.items():
        if df is not None and not df.empty:
            df.to_excel(writer, sheet_name=sn, index=False)

wb = load_workbook(output_path)

if f"{S}_Plan_By_Machine"      in wb.sheetnames: style_machine_wise(wb[f"{S}_Plan_By_Machine"])
if f"{S}_Production_vs_Indent" in wb.sheetnames: style_pvsi(wb[f"{S}_Production_vs_Indent"])
if f"{S}_Multi_Machine_Parts"  in wb.sheetnames: style_multi_machine(wb[f"{S}_Multi_Machine_Parts"])
if f"{S}_Inventory_Target"     in wb.sheetnames: style_inv_target(wb[f"{S}_Inventory_Target"])
if f"{S}_CO_Queue"             in wb.sheetnames: style_co_queue(wb[f"{S}_CO_Queue"])
if f"{S}_Fixed_Adherence"      in wb.sheetnames: style_fixed_adherence(wb[f"{S}_Fixed_Adherence"])

for sn, hx in HEADER_COLORS.items():
    if sn in wb.sheetnames and sn not in (
        f"{S}_Plan_By_Machine", f"{S}_Production_vs_Indent",
        f"{S}_Multi_Machine_Parts", f"{S}_Inventory_Target",
        f"{S}_CO_Queue", f"{S}_Fixed_Adherence",
    ):
        style_sheet(wb[sn], hx)

# Daily indent status YES/NO colouring
if f"{S}_Daily_Indent_Status" in wb.sheetnames:
    ws_is      = wb[f"{S}_Daily_Indent_Status"]
    headers_is = [c.value for c in ws_is[1]]
    mc_is      = headers_is.index("Meets_Daily_Indent") + 1 if "Meets_Daily_Indent" in headers_is else None
    cv_is      = headers_is.index("Covers_With_Inv")    + 1 if "Covers_With_Inv"    in headers_is else None
    for row in ws_is.iter_rows(min_row=2):
        if mc_is:
            cell = row[mc_is-1]
            cell.fill = (
                PatternFill("solid", fgColor="C6EFCE") if str(cell.value) == "YES ✓"
                else PatternFill("solid", fgColor="FFC7CE")
            )
        if cv_is:
            cell = row[cv_is-1]
            cell.fill = (
                PatternFill("solid", fgColor="C6EFCE") if str(cell.value) == "YES ✓"
                else PatternFill("solid", fgColor="FFEB9C")
            )
        if mc_is and str(row[mc_is-1].value) == "NO ✗":
            for cell in row: cell.font = Font(bold=True)

for sn, hx in HEADER_COLORS.items():
    if sn in wb.sheetnames:
        wb[sn].sheet_properties.tabColor = hx

wb.save(output_path)
print(f"  Formatting applied  ✓")

# =============================================================
# SECTION 26 — FINAL SUMMARY
# =============================================================

print(f"\n{'='*65}")
print(f"  Smart APS V9 — Section: {SECTION_LABEL}  —  {PLANNING_DATE}")
print(f"  Safety: {SAFETY_DAYS}d  |  Target: {TARGET_DAYS}d  |  Purge: {int(COLOR_PURGE_HRS*60)}min")
print(f"{'='*65}")

for s, cnt in audit_df["Status"].value_counts().items():
    print(f"  {'✓' if s=='ENTERS SCHEDULER' else '·'} {s:<50}: {cnt:>4}")

print(f"\n  Results:")
print(f"    Plan rows    : {len(plan_df):>4}")
print(f"    Not planned  : {len(not_df):>4}")
print(f"    Deferred     : {len(def_df):>4}")

if not fixed_adh_df.empty:
    fa_u = (fixed_adh_df["Status"] == "FIXED MACHINE USED").sum()
    fa_f = fixed_adh_df["Status"].str.contains("FALLBACK", na=False).sum()
    fa_n = (fixed_adh_df["Status"] == "NOT PLANNED TODAY").sum()
    print(f"\n  Fixed Machine Adherence:")
    print(f"    Fixed used correctly : {fa_u}")
    print(f"    Fallback used        : {fa_f}")
    print(f"    Not planned          : {fa_n}")

if not disp_df.empty:
    dp = (disp_df["Result"] == "PLACED").sum()
    df_ = disp_df["Result"].str.contains("FAILED", na=False).sum()
    print(f"\n  Displacement: {dp} placed  |  {df_} failed")

if not inv_tgt_df.empty:
    for s in ["AT_TARGET","BUILDING","BELOW_SAFETY","CRITICAL"]:
        print(f"    {s:<20}: {(inv_tgt_df['Buffer_Status']==s).sum():>4}")

if not mach_df.empty:
    print(f"\n  Machine utilization:")
    print(f"    Average     : {mach_df['Utilization_%'].mean():.1f}%")
    print(f"    UNDERUSED   : {(mach_df['Status']=='UNDERUSED').sum()}")
    print(f"    Purges      : {mach_df['Color_Purges'].sum()}")

if not vt_terminal_status_df.empty:
    zt = (vt_terminal_status_df["Available_Today"] == "NO — ZERO STOCK").sum()
    bt = int(vt_terminal_status_df["Parts_Blocked_Count"].sum())
    print(f"\n  Terminals: {zt} zero  |  {bt} parts blocked")

print(f"\n  Output → {output_path}")
print(f"  State  → {MACHINE_STATE_FILE}")
print(f"\n  UPDATE DAILY: PLANNING_DATE = date({PLANNING_DATE.year}, "
      f"{PLANNING_DATE.month}, {PLANNING_DATE.day + 1})")
print(f"{'='*65}")